# 💊 OTC Safe-Dose Assistant
### A safety-first, rule-based medicine-advice chatbot — fully explained notebook

This notebook builds, tests, evaluates and launches a chatbot that suggests **over-the-counter (OTC)
medicines and doses** for common adult complaints, such as headache, fever, mild pain, cold,
hay fever, heartburn and mild diarrhoea.

> ⚠️ **Research prototype — not a medical device.** All doses are placeholders marked `VERIFY`. A pharmacist must check
> each one against the official product label before any real use.

---

## 📑 Contents

| Part | What happens | Required? |
|---|---|---|
| **0** | Set up Colab: install packages, create the project folder | ✅ |
| **1** | The **knowledge base**: drug formulary + red-flag triage questions (JSON) | ✅ |
| **2** | The **rules engine**: Rules 1–6, the "brain" that decides | ✅ |
| **3** | The **safety validator** and **red alarm**: independent double-check | ✅ |
| **4** | Try the engine on example patients, including a deliberate fault | Optional demo |
| **5** | Optional **AI intake** and **English/Urdu** text | ✅ |
| **6** | The **web interface** (Streamlit GUI) | ✅ |
| **7** | **Unit tests**: 60 automatic checks | Recommended |
| **8** | **Evaluation** with penalty scoring + fault injection | Recommended |
| **9** | **Launch** the chatbot and get a public link | ✅ |
| **10** | Troubleshooting, downloads, limitations | As needed |

**▶ How to run:** click **Runtime → Run all**. Then scroll to **Part 9** and click the green link.

---

## 🧠 How the system works (big picture)

```
 Patient answers                                                                What the user sees
 (guided form, or optional         ┌──────────────┐      ┌────────────────┐
  AI pre-fill the user confirms) ─►│  rules.py    │─────►│ validator.py   │──┬─► ✅ Recommendation card
                                   │  Rules 1–6   │      │ checks V0–V13  │  ├─► 🚨 Emergency screen (red flag)
                                   │ (reads JSON  │      │ (independent   │  ├─► 🧑‍⚕️ "See a pharmacist/doctor"
                                   │  formulary)  │      │  re-check)     │  └─► 🔴 RED ALARM if any check fails
                                   └──────────────┘      └────────────────┘          + incident written to incidents.csv
```

**Three key design ideas:**
1. **The AI never chooses a drug or a dose.** Every drug name and every number comes from a curated file (`formulary.json`).
2. **Deterministic rules run in a fixed order** and stop at the first exit. For example, a red flag stops everything before any medicine is discussed.
3. **Defence in depth:** a second, independently written program (the validator) re-checks every answer. If anything is unsafe, the answer is blocked and logged.

---
# Part 0 — Setup

### Cell 0.1 — Install packages and create the project folder

| Line | What it does |
|---|---|
| `PROJECT_DIR = ...` | The folder where every project file will be written |
| `os.makedirs(...)` | Creates the folder, plus `tests/` for unit tests and `.streamlit/` for app settings |
| `%cd` | Moves the notebook into that folder, so the `%%writefile` cells below save files there |
| `pip install` | Installs **Streamlit** (the web-app framework), **requests** (used by the optional AI intake) and **pytest** (the test runner). `-q` keeps the output short. |

In [ ]:
import os

# Folder that will hold the whole project
PROJECT_DIR = "/content/otc-safe-dose"

# Create the project folder and its two sub-folders (no error if they already exist)
os.makedirs(f"{PROJECT_DIR}/tests", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/.streamlit", exist_ok=True)

# Work inside the project folder from now on
%cd {PROJECT_DIR}

# Install the three libraries the project needs
!pip install -q "streamlit>=1.40" "requests>=2.31" "pytest>=8.0"
print("✅ Setup done")

### Cell 0.2 — `requirements.txt` and Streamlit settings

**What is `%%writefile`?** It is a Colab "magic" command. Everything in the cell below that first line
is **saved as a file** instead of being run. Most cells in this notebook use it to create one project file each.
`%%writefile -a` **appends** to a file, which lets us split long files across several explained cells.

* `requirements.txt` lists the libraries, so anyone can reinstall them with `pip install -r requirements.txt`.
* `.streamlit/config.toml` sets the app's teal accent colour (`primaryColor`), runs it without opening a browser
  (`headless`), and switches off Streamlit's anonymous usage statistics.

In [ ]:
%%writefile requirements.txt
streamlit>=1.40
requests>=2.31
pytest>=8.0


In [ ]:
%%writefile .streamlit/config.toml
[theme]
primaryColor = "#0f766e"
font = "sans serif"

[server]
headless = true

[browser]
gatherUsageStats = false


---
# Part 1 — The knowledge base (what the chatbot "knows")

The medical knowledge is kept **outside the code**, in two JSON files. Keeping it there means:
* a pharmacist can review or correct doses **without reading Python**;
* the code can never invent a drug, because it can only use what is listed here.

### Cell 1.1 — `formulary.json` (the list of allowed medicines)

Each medicine entry contains:

| Field | Meaning | Example (paracetamol) |
|---|---|---|
| `strength_mg`, `form`, `unit` | What one tablet/capsule contains | 500 mg tablet |
| `single_dose_min_mg` / `single_dose_max_mg` | Smallest and largest dose taken at one time | 500–1000 mg |
| `min_interval_h`, `interval_text` | Minimum hours between doses, and how it is shown to the user | 4 h ("every 4–6 hours") |
| `max_24h_mg` | **Hard daily ceiling** | 4000 mg |
| `reduced_limits` | Lower ceilings for higher-risk patients: regular alcohol use, or weight under 50 kg (15 mg/kg per dose, 60 mg/kg per day) | 3000 mg |
| `first_dose_mg` / `subsequent_dose_mg` | Used only by loperamide (4 mg first, then 2 mg after each loose stool) | — |
| `indications` | Which complaints this medicine may be offered for | headache, fever… |
| `contraindications` | Conditions, interacting medicines and allergies that **exclude** this medicine | liver disease, paracetamol allergy |
| `avoid_if_age_gte` | Age from which self-care use is not offered (ibuprofen and cetirizine: 65) | — |
| `warnings`, `interactions`, `administration` | Text shown on the recommendation card | — |
| `source`, `verified_by`, `verified_date`, `status` | **Audit trail.** `status` stays `VERIFY` until a pharmacist signs the entry off | VERIFY |

The five medicines are paracetamol, ibuprofen, cetirizine, loperamide and calcium carbonate.

In [ ]:
%%writefile formulary.json
{
  "_meta": {
    "version": "0.1.0",
    "status": "UNVERIFIED PROTOTYPE DATA",
    "instructions": "Every entry is a PLACEHOLDER marked VERIFY. Before any public use, a registered pharmacist must check every number, contraindication and warning against the official label of the exact product you dispense (e.g., DailyMed SPL, BNF monograph, or the DRAP-approved label), then fill in 'source', 'verified_by', 'verified_date' and set 'status' to 'VERIFIED'.",
    "formulary_verification_date": null
  },
  "drugs": {
    "paracetamol": {
      "display_name": "Paracetamol (acetaminophen)",
      "ingredient": "paracetamol",
      "drug_class": "analgesic / antipyretic",
      "form": "tablet",
      "unit": "tablet",
      "strength_mg": 500,
      "single_dose_min_mg": 500,
      "single_dose_max_mg": 1000,
      "min_interval_h": 4,
      "interval_text": "every 4-6 hours if needed",
      "max_24h_mg": 4000,
      "reduced_limits": {
        "applies_if": ["alcohol_regular", "low_weight"],
        "max_24h_mg": 3000,
        "low_weight_threshold_kg": 50,
        "per_kg_single_mg": 15,
        "per_kg_24h_mg": 60
      },
      "indications": ["headache", "fever", "musculoskeletal_pain", "common_cold"],
      "contraindications": {
        "conditions": ["liver_disease"],
        "med_categories": [],
        "allergies": ["paracetamol"]
      },
      "avoid_if_age_gte": null,
      "administration": "Swallow with water, with or without food.",
      "interactions": [
        "Many cold/flu, sinus and 'extra' pain products also contain paracetamol - count them in the 24-hour total.",
        "Warfarin: regular paracetamol use can raise INR - ask a pharmacist."
      ],
      "warnings": [
        "Do not take with any other product that contains paracetamol.",
        "Taking too much can cause serious liver damage even if you feel well. If you have taken more than the maximum, get medical help immediately.",
        "Avoid alcohol while taking this medicine."
      ],
      "source": "VERIFY against the official product label (DailyMed / BNF / DRAP-approved label).",
      "verified_by": "VERIFY",
      "verified_date": null,
      "status": "VERIFY"
    },
    "ibuprofen": {
      "display_name": "Ibuprofen",
      "ingredient": "ibuprofen",
      "drug_class": "NSAID (anti-inflammatory painkiller)",
      "form": "tablet",
      "unit": "tablet",
      "strength_mg": 200,
      "single_dose_min_mg": 200,
      "single_dose_max_mg": 400,
      "min_interval_h": 6,
      "interval_text": "every 6-8 hours if needed",
      "max_24h_mg": 1200,
      "reduced_limits": null,
      "indications": ["headache", "fever", "musculoskeletal_pain", "common_cold"],
      "contraindications": {
        "conditions": [
          "peptic_ulcer_or_gi_bleed",
          "kidney_disease",
          "asthma_nsaid_sensitive",
          "heart_failure",
          "high_blood_pressure",
          "inflammatory_bowel_disease"
        ],
        "med_categories": [
          "anticoagulant",
          "antiplatelet",
          "nsaid",
          "oral_corticosteroid",
          "ssri_snri",
          "lithium",
          "methotrexate",
          "bp_diuretic_acei_arb"
        ],
        "allergies": ["ibuprofen", "nsaid_aspirin"]
      },
      "avoid_if_age_gte": 65,
      "administration": "Take with or just after food, with a full glass of water.",
      "interactions": [
        "Do not combine with other NSAIDs (aspirin, diclofenac, naproxen, mefenamic acid).",
        "Increases bleeding risk with blood thinners, antiplatelets, SSRIs/SNRIs and steroids."
      ],
      "warnings": [
        "Stop and get medical help if you notice black stools, vomiting blood or severe stomach pain.",
        "Do not take with other anti-inflammatory painkillers.",
        "Not for use in pregnancy."
      ],
      "source": "VERIFY against the official product label (DailyMed / BNF / DRAP-approved label).",
      "verified_by": "VERIFY",
      "verified_date": null,
      "status": "VERIFY"
    },
    "cetirizine": {
      "display_name": "Cetirizine",
      "ingredient": "cetirizine",
      "drug_class": "non-sedating antihistamine",
      "form": "tablet",
      "unit": "tablet",
      "strength_mg": 10,
      "single_dose_min_mg": 10,
      "single_dose_max_mg": 10,
      "min_interval_h": 24,
      "interval_text": "once a day",
      "max_24h_mg": 10,
      "reduced_limits": null,
      "indications": ["allergic_rhinitis"],
      "contraindications": {
        "conditions": ["kidney_disease"],
        "med_categories": [],
        "allergies": ["cetirizine_hydroxyzine"]
      },
      "avoid_if_age_gte": 65,
      "administration": "Swallow with water, with or without food.",
      "interactions": [
        "Alcohol and other sedating medicines can increase drowsiness."
      ],
      "warnings": [
        "May cause drowsiness - do not drive or operate machinery if affected.",
        "Avoid alcohol."
      ],
      "source": "VERIFY against the official product label (DailyMed / BNF / DRAP-approved label).",
      "verified_by": "VERIFY",
      "verified_date": null,
      "status": "VERIFY"
    },
    "loperamide": {
      "display_name": "Loperamide",
      "ingredient": "loperamide",
      "drug_class": "antidiarrhoeal",
      "form": "capsule",
      "unit": "capsule",
      "strength_mg": 2,
      "single_dose_min_mg": 2,
      "single_dose_max_mg": 4,
      "first_dose_mg": 4,
      "subsequent_dose_mg": 2,
      "min_interval_h": 0,
      "interval_text": "first dose 4 mg, then 2 mg after each loose stool",
      "max_24h_mg": 8,
      "reduced_limits": null,
      "indications": ["diarrhoea"],
      "contraindications": {
        "conditions": ["inflammatory_bowel_disease"],
        "med_categories": [],
        "allergies": ["loperamide"]
      },
      "avoid_if_age_gte": null,
      "administration": "Swallow with water. Drinking fluids / oral rehydration salts (ORS) is the most important treatment.",
      "interactions": [
        "Never take more than the label maximum - high doses can cause dangerous heart rhythm problems."
      ],
      "warnings": [
        "Drink plenty of fluids or ORS to prevent dehydration.",
        "Stop and see a doctor if diarrhoea lasts more than 48 hours, or if blood, fever or a swollen tummy appears.",
        "Stop if you become constipated."
      ],
      "source": "VERIFY against the official product label (DailyMed / BNF / DRAP-approved label).",
      "verified_by": "VERIFY",
      "verified_date": null,
      "status": "VERIFY"
    },
    "calcium_carbonate": {
      "display_name": "Calcium carbonate (antacid)",
      "ingredient": "calcium_carbonate",
      "drug_class": "antacid",
      "form": "chewable tablet",
      "unit": "chewable tablet",
      "strength_mg": 500,
      "single_dose_min_mg": 500,
      "single_dose_max_mg": 1000,
      "min_interval_h": 1,
      "interval_text": "when symptoms occur (at least 1 hour apart)",
      "max_24h_mg": 7500,
      "reduced_limits": null,
      "indications": ["heartburn"],
      "contraindications": {
        "conditions": ["kidney_disease"],
        "med_categories": [],
        "allergies": ["calcium_carbonate"]
      },
      "avoid_if_age_gte": null,
      "administration": "Chew thoroughly before swallowing.",
      "interactions": [
        "Can reduce absorption of other medicines (e.g., some antibiotics, iron, thyroid tablets) - take them at least 2 hours apart."
      ],
      "warnings": [
        "Take other medicines at least 2 hours apart from this antacid.",
        "See a doctor if you need an antacid for more than 14 days."
      ],
      "source": "VERIFY against the official product label (DailyMed / BNF / DRAP-approved label).",
      "verified_by": "VERIFY",
      "verified_date": null,
      "status": "VERIFY"
    }
  }
}


### Cell 1.2 — `triage.json` (red flags and self-care limits)

This file holds **Rule 1 (triage)** and **Rule 5 (duration)** data:

* `emergency_number`: shown on emergency screens (default **1122**, Rescue Pakistan). It can be changed without touching code.
* `general_red_flags`: danger signs asked for **every** complaint, such as chest pain, breathing difficulty, collapse and anaphylaxis.
* `complaints`: for each of the 7 complaints:
  * `candidates`: which formulary medicines may be offered, in order of preference;
  * `red_flags`: complaint-specific danger signs, each with a `severity`:
    * `emergency` → full red screen and "call 1122 now";
    * `urgent_doctor` → orange screen and "see a doctor today";
  * `max_self_care_days`: how long self-treatment is acceptable. Longer than this, the patient is referred to a doctor;
  * `stop_advice`: the "when to stop and see a doctor" sentence shown with every recommendation.

Note the Pakistan-specific **dengue warning signs** under *fever*.

In [ ]:
%%writefile triage.json
{
  "_meta": {
    "version": "0.1.0",
    "status": "UNVERIFIED PROTOTYPE DATA - red flags and self-care limits must be reviewed by a clinician before use.",
    "severity_levels": {
      "emergency": "Call the emergency number / go to an emergency department now.",
      "urgent_doctor": "Do not self-treat; see a doctor today."
    }
  },
  "emergency_number": "1122",
  "emergency_label": "Rescue 1122 (Pakistan)",
  "general_red_flags": [
    {"id": "chest_pain", "text": "Chest pain, pressure or tightness", "severity": "emergency"},
    {"id": "breathing_difficulty", "text": "Difficulty breathing or shortness of breath at rest", "severity": "emergency"},
    {"id": "collapse_drowsy", "text": "Fainting, collapse, or very drowsy / hard to wake", "severity": "emergency"},
    {"id": "anaphylaxis", "text": "Swelling of the face, lips or tongue, or a widespread rash with feeling very unwell", "severity": "emergency"},
    {"id": "feels_seriously_ill", "text": "You feel extremely unwell or something feels seriously wrong", "severity": "urgent_doctor"}
  ],
  "complaints": {
    "headache": {
      "label": "Headache",
      "candidates": ["paracetamol", "ibuprofen"],
      "max_self_care_days": 3,
      "stop_advice": "Stop self-treatment and see a doctor if the headache lasts more than 3 days, keeps coming back, gets worse, or you need painkillers on more than 2-3 days a week.",
      "red_flags": [
        {"id": "thunderclap", "text": "Sudden, severe 'worst ever' headache that reached full strength within about a minute", "severity": "emergency"},
        {"id": "fever_stiff_neck_rash", "text": "Headache with fever AND a stiff neck, or a rash that does not fade when pressed", "severity": "emergency"},
        {"id": "head_injury", "text": "Headache after a head injury", "severity": "emergency"},
        {"id": "neuro_deficit", "text": "Weakness, numbness, drooping face, slurred speech or difficulty walking", "severity": "emergency"},
        {"id": "confusion_seizure", "text": "Confusion, change in behaviour, or a seizure (fit)", "severity": "emergency"},
        {"id": "vision_change", "text": "Loss of vision, double vision, or a painful red eye", "severity": "urgent_doctor"},
        {"id": "pregnant_postpartum", "text": "You are pregnant or gave birth in the last 6 weeks", "severity": "urgent_doctor"},
        {"id": "new_over_50", "text": "You are over 50 and this is a new or different type of headache", "severity": "urgent_doctor"},
        {"id": "worse_lying_vomiting", "text": "Headache that wakes you, gets worse lying down or coughing, or comes with repeated vomiting", "severity": "urgent_doctor"}
      ]
    },
    "fever": {
      "label": "Fever",
      "candidates": ["paracetamol", "ibuprofen"],
      "max_self_care_days": 3,
      "stop_advice": "See a doctor if the fever lasts more than 3 days, goes above 40 C, or you get new symptoms. In dengue season, get tested if fever persists.",
      "red_flags": [
        {"id": "very_high_temp", "text": "Temperature of 40 C (104 F) or higher", "severity": "urgent_doctor"},
        {"id": "stiff_neck_rash", "text": "Stiff neck, or a rash that does not fade when pressed", "severity": "emergency"},
        {"id": "dengue_warning", "text": "Bleeding gums or nose, blood in vomit or stool, unusual bruising, severe tummy pain or repeated vomiting", "severity": "emergency"},
        {"id": "confusion_seizure", "text": "Confusion or a seizure (fit)", "severity": "emergency"},
        {"id": "dehydration", "text": "Cannot keep fluids down, or passing very little urine", "severity": "urgent_doctor"},
        {"id": "immunocompromised", "text": "You are on chemotherapy, long-term steroid tablets, or have a weakened immune system", "severity": "urgent_doctor"}
      ]
    },
    "musculoskeletal_pain": {
      "label": "Mild muscle / joint / back pain",
      "candidates": ["paracetamol", "ibuprofen"],
      "max_self_care_days": 5,
      "stop_advice": "See a doctor if the pain lasts more than 5 days, gets worse, or stops you from normal activities.",
      "red_flags": [
        {"id": "injury_deformity", "text": "After a fall or injury, with deformity, or you cannot bear weight or move the limb", "severity": "urgent_doctor"},
        {"id": "hot_swollen_joint", "text": "A hot, red, swollen joint, especially with fever", "severity": "urgent_doctor"},
        {"id": "back_neuro", "text": "Back pain with leg weakness or numbness, numbness around the genitals or buttocks, or new bladder/bowel problems", "severity": "emergency"},
        {"id": "cardiac_pattern", "text": "Pain in the chest, left arm, jaw or upper back with sweating or breathlessness", "severity": "emergency"},
        {"id": "calf_swelling", "text": "Pain in one calf with swelling, warmth or redness", "severity": "urgent_doctor"}
      ]
    },
    "common_cold": {
      "label": "Common cold symptoms (aches, sore throat, mild fever)",
      "candidates": ["paracetamol", "ibuprofen"],
      "max_self_care_days": 7,
      "stop_advice": "See a doctor if symptoms last more than 7 days, get worse after improving, or you develop high fever, ear or face pain.",
      "red_flags": [
        {"id": "high_fever_3d", "text": "Fever above 39 C lasting more than 3 days", "severity": "urgent_doctor"},
        {"id": "coughing_blood", "text": "Coughing up blood", "severity": "urgent_doctor"},
        {"id": "wheeze", "text": "Wheezing or noisy breathing", "severity": "urgent_doctor"},
        {"id": "worse_after_better", "text": "Symptoms improved and then suddenly got much worse", "severity": "urgent_doctor"}
      ]
    },
    "allergic_rhinitis": {
      "label": "Hay fever / allergic runny nose",
      "candidates": ["cetirizine"],
      "max_self_care_days": 14,
      "stop_advice": "See a doctor or pharmacist if there is no improvement after 14 days, or symptoms affect your sleep or breathing.",
      "red_flags": [
        {"id": "wheeze_asthma", "text": "Wheezing, or your asthma is getting worse", "severity": "urgent_doctor"},
        {"id": "eye_pain_vision", "text": "Eye pain or changes in vision", "severity": "urgent_doctor"},
        {"id": "one_sided_bloody", "text": "Blocked on one side only, with blood or a foul-smelling discharge", "severity": "urgent_doctor"}
      ]
    },
    "heartburn": {
      "label": "Heartburn / indigestion",
      "candidates": ["calcium_carbonate"],
      "max_self_care_days": 14,
      "stop_advice": "See a doctor if you need an antacid for more than 14 days, or symptoms keep returning.",
      "red_flags": [
        {"id": "swallowing", "text": "Difficulty or pain when swallowing", "severity": "urgent_doctor"},
        {"id": "gi_bleed", "text": "Vomiting blood or 'coffee grounds', or black tarry stools", "severity": "emergency"},
        {"id": "weight_loss", "text": "Weight loss you cannot explain", "severity": "urgent_doctor"},
        {"id": "cardiac_pattern", "text": "Chest pain spreading to the arm, jaw or neck, or with sweating or breathlessness", "severity": "emergency"},
        {"id": "new_over_55", "text": "You are over 55 and this is new heartburn", "severity": "urgent_doctor"},
        {"id": "persistent_vomiting", "text": "Repeated vomiting", "severity": "urgent_doctor"}
      ]
    },
    "diarrhoea": {
      "label": "Mild diarrhoea",
      "candidates": ["loperamide"],
      "max_self_care_days": 2,
      "stop_advice": "Keep drinking fluids / ORS. See a doctor if diarrhoea lasts more than 48 hours, or blood, fever or severe pain appears.",
      "red_flags": [
        {"id": "blood_stool", "text": "Blood or mucus in the stool, or black stools", "severity": "urgent_doctor"},
        {"id": "fever", "text": "Fever of 38 C or higher", "severity": "urgent_doctor"},
        {"id": "severe_abdominal_pain", "text": "Severe tummy pain or a swollen tummy", "severity": "urgent_doctor"},
        {"id": "dehydration", "text": "Signs of dehydration: very little or dark urine, dizziness on standing, very dry mouth", "severity": "urgent_doctor"},
        {"id": "after_antibiotics_hospital", "text": "Started after taking antibiotics or a hospital stay", "severity": "urgent_doctor"}
      ]
    }
  }
}


---
# Part 2 — The rules engine (`rules.py`)

This is the chatbot's decision-maker. It is **deterministic**: the same answers always give the same output,
with no randomness and no AI. The file is split over 10 cells so each part can be explained.

### Cell 2.1 — Header, constants and option lists
* The **docstring** (text in triple quotes) summarises the six rules and their possible outcomes.
* `DISCLAIMER`: the sentence every recommendation must carry. The validator checks that it is present.
* `ACTIONS`: the only five outcomes the engine may return: `ASK` (an answer is missing), `EMERGENCY`, `URGENT_DOCTOR`, `REFER`, `RECOMMEND`.
* `CONDITION_OPTIONS`, `MED_CATEGORY_OPTIONS`, `ALLERGY_OPTIONS`: the choices shown in the GUI's drop-down lists.
  Their **keys** (for example `"anticoagulant"`) are the same words used in `formulary.json` contraindications, which is how the two connect.
* `TRIAGE_FIELDS`, `PROFILE_FIELDS`, `HISTORY_FIELDS`: which answers belong to which step of the interface.

**Convention:** a value of `None` means *"not answered yet"*. An empty list `[]` means *"answered: none"*. The engine never treats
"unknown" as "no".

In [ ]:
%%writefile rules.py
"""
rules.py - Deterministic decision engine for the OTC Safe-Dose Assistant.

The engine never invents drugs or doses. Every drug name, strength, dose,
interval and 24-hour maximum is read from formulary.json; every red-flag
question and self-care duration limit is read from triage.json.

Decision rules run strictly in order and stop at the first exit:
    Rule 1  Red-flag triage           -> EMERGENCY / URGENT_DOCTOR / ASK
    Rule 2  Demographics and scope    -> ASK / REFER
    Rule 3  Contraindications         -> ASK / REFER
    Rule 4  24-hour cap and interval  -> ASK / REFER
    Rule 5  Self-care duration limit  -> REFER
    Rule 6  Build the recommendation  -> RECOMMEND

A patient is a plain dict. A value of None means "not answered yet";
an empty list means "answered: none".
"""
from __future__ import annotations

import json
import math
import os
from functools import lru_cache
from pathlib import Path
from typing import Any

APP_VERSION = "0.1.0"
BASE_DIR = Path(__file__).resolve().parent
DISCLAIMER = ("This is general information, not a diagnosis. "
              "Confirm with a pharmacist or doctor.")

ACTIONS = ("ASK", "EMERGENCY", "URGENT_DOCTOR", "REFER", "RECOMMEND")

# Option lists shown in the GUI. Keys are referenced by formulary.json.
CONDITION_OPTIONS = {
    "kidney_disease": "Long-term kidney disease",
    "liver_disease": "Long-term liver disease (e.g., cirrhosis, hepatitis)",
    "peptic_ulcer_or_gi_bleed": "Stomach ulcer, or stomach/bowel bleeding (ever)",
    "asthma_nsaid_sensitive": "Asthma that gets worse with aspirin or ibuprofen",
    "heart_failure": "Heart failure",
    "high_blood_pressure": "High blood pressure",
    "inflammatory_bowel_disease": "Crohn's disease or ulcerative colitis",
}
MED_CATEGORY_OPTIONS = {
    "anticoagulant": "Blood thinner (e.g., warfarin, rivaroxaban, apixaban)",
    "antiplatelet": "Antiplatelet (e.g., low-dose aspirin, clopidogrel)",
    "nsaid": "Another anti-inflammatory painkiller (e.g., diclofenac, naproxen, mefenamic acid)",
    "oral_corticosteroid": "Steroid tablets (e.g., prednisolone)",
    "ssri_snri": "Antidepressant SSRI/SNRI (e.g., sertraline, fluoxetine)",
    "lithium": "Lithium",
    "methotrexate": "Methotrexate",
    "bp_diuretic_acei_arb": "Blood-pressure tablets: ACE inhibitor, ARB or diuretic",
    "other": "Other regular medicine",
}
ALLERGY_OPTIONS = {
    "paracetamol": "Paracetamol",
    "ibuprofen": "Ibuprofen",
    "nsaid_aspirin": "Aspirin or other NSAIDs",
    "cetirizine_hydroxyzine": "Cetirizine or hydroxyzine",
    "loperamide": "Loperamide",
    "calcium_carbonate": "Calcium carbonate / antacids",
}

TRIAGE_FIELDS = ("complaint", "symptom_days")
PROFILE_FIELDS = ("age", "sex", "weight_kg", "pregnant", "breastfeeding")
HISTORY_FIELDS = ("conditions", "current_meds", "allergies",
                  "alcohol_regular", "num_regular_meds")
SEXES = ("female", "male", "other")




### Cell 2.2 — The `Knowledge` class: read-only access to the JSON files
* `__init__` loads `formulary.json` and `triage.json`. Tests can pass in modified copies instead.
* Small helper properties keep the rest of the code readable: `kb.drugs`, `kb.complaints`, `kb.emergency_number`
  (which can also be overridden with the environment variable `MEDBOT_EMERGENCY_NUMBER`).
* `red_flags_for(complaint)` returns **general + complaint-specific** red flags together.
* `default_knowledge()` loads the files once and caches them (`lru_cache`), so they are not re-read on every click.

In [ ]:
%%writefile -a rules.py
# --------------------------------------------------------------------------
# Knowledge base
# --------------------------------------------------------------------------
def _load(name: str) -> dict:
    with open(BASE_DIR / name, encoding="utf-8") as fh:
        return json.load(fh)


class Knowledge:
    """Read-only access to formulary.json and triage.json."""

    def __init__(self, formulary: dict | None = None, triage: dict | None = None):
        self.formulary = formulary if formulary is not None else _load("formulary.json")
        self.triage = triage if triage is not None else _load("triage.json")

    @property
    def drugs(self) -> dict:
        return self.formulary["drugs"]

    @property
    def complaints(self) -> dict:
        return self.triage["complaints"]

    @property
    def emergency_number(self) -> str:
        return os.environ.get("MEDBOT_EMERGENCY_NUMBER") or self.triage["emergency_number"]

    @property
    def verification_date(self) -> str | None:
        return self.formulary["_meta"].get("formulary_verification_date")

    def red_flags_for(self, complaint: str) -> list[dict]:
        return list(self.triage["general_red_flags"]) + list(self.complaints[complaint]["red_flags"])

    def candidates_for(self, complaint: str) -> list[str]:
        return list(self.complaints[complaint]["candidates"])

    def max_self_care_days(self, complaint: str) -> int:
        return self.complaints[complaint]["max_self_care_days"]


@lru_cache(maxsize=1)
def default_knowledge() -> Knowledge:
    return Knowledge()




### Cell 2.3 — Helper functions (the dose arithmetic)
| Function | Job |
|---|---|
| `stage_of_field` | Tells the GUI which step a missing answer belongs to (triage, profile or safety) |
| `floor_to` | Rounds **down** to whole tablets. For example, 675 mg with 500 mg tablets → 500 mg. Rounding down is the safe direction. |
| `effective_limits` | Starts from the formulary limits, then applies **patient-specific reductions**: regular alcohol → 3000 mg/day paracetamol ceiling; weight < 50 kg → 15 mg/kg per dose and 60 mg/kg per day |
| `contraindication_reasons` | Compares the patient's conditions, medicines, allergies and age with the drug's contraindications, and returns plain-English reasons |
| `prior_intake_for` | **Adds up every product** containing an ingredient in the last 24 h. This is what catches *hidden paracetamol* in cold & flu products. It also finds the time of the most recent dose. |
| `scope_reasons_demographic` / `scope_reasons_history` | Out-of-scope patients who must be referred, not dosed: under 18, pregnant, breastfeeding, kidney or liver disease, age 75+ with ≥ 2 regular medicines |

In [ ]:
%%writefile -a rules.py
# --------------------------------------------------------------------------
# Helpers
# --------------------------------------------------------------------------
def stage_of_field(field: str) -> str:
    """Map a missing field to the GUI stage that collects it."""
    if field in TRIAGE_FIELDS or field.startswith("red_flags."):
        return "triage"
    if field in PROFILE_FIELDS:
        return "profile"
    return "safety"


def floor_to(value: float, step: int) -> int:
    """Round down to a whole number of tablets/capsules."""
    return int(math.floor(value / step + 1e-9) * step)


def _is_num(x: Any) -> bool:
    return isinstance(x, (int, float)) and not isinstance(x, bool) and not math.isnan(x)


def effective_limits(drug: dict, patient: dict) -> dict:
    """Single-dose and 24-hour limits after patient-specific reductions."""
    s = drug["strength_mg"]
    single_min = drug["single_dose_min_mg"]
    single_max = drug["single_dose_max_mg"]
    max_24h = drug["max_24h_mg"]
    notes: list[str] = []
    red = drug.get("reduced_limits")
    if red:
        weight = patient.get("weight_kg")
        low_weight = _is_num(weight) and weight < red.get("low_weight_threshold_kg", 0)
        triggers = []
        if patient.get("alcohol_regular") and "alcohol_regular" in red["applies_if"]:
            triggers.append("regular alcohol use")
        if low_weight and "low_weight" in red["applies_if"]:
            triggers.append(f"body weight under {red['low_weight_threshold_kg']} kg")
        if triggers:
            max_24h = min(max_24h, red["max_24h_mg"])
            notes.append("Lower daily maximum because of " + " and ".join(triggers) + ".")
        if low_weight:
            single_max = min(single_max, floor_to(red["per_kg_single_mg"] * weight, s))
            max_24h = min(max_24h, floor_to(red["per_kg_24h_mg"] * weight, s))
            notes.append("Dose limited by body weight.")
    return {"single_min": single_min, "single_max": single_max,
            "max_24h": max_24h, "notes": notes}


def contraindication_reasons(drug: dict, patient: dict) -> list[str]:
    ci = drug.get("contraindications") or {}
    reasons = []
    for c in patient.get("conditions") or []:
        if c in ci.get("conditions", []):
            reasons.append(f"Not suitable with: {CONDITION_OPTIONS.get(c, c)}")
    for m in patient.get("current_meds") or []:
        if m in ci.get("med_categories", []):
            reasons.append(f"Interacts with: {MED_CATEGORY_OPTIONS.get(m, m)}")
    for a in patient.get("allergies") or []:
        if a in ci.get("allergies", []):
            reasons.append(f"Allergy: {ALLERGY_OPTIONS.get(a, a)}")
    limit_age = drug.get("avoid_if_age_gte")
    if limit_age is not None and _is_num(patient.get("age")) and patient["age"] >= limit_age:
        reasons.append(f"Not for self-care at age {limit_age}+ without professional advice")
    return reasons


def prior_intake_for(patient: dict, ingredient: str) -> tuple[float, float | None]:
    """Total mg taken in the last 24 h and hours since the most recent dose."""
    entries = [e for e in (patient.get("prior_intake") or [])
               if e.get("ingredient") == ingredient and (e.get("total_mg_24h") or 0) > 0]
    total = sum(e["total_mg_24h"] for e in entries)
    hours = [e["hours_since_last"] for e in entries if _is_num(e.get("hours_since_last"))]
    return total, (min(hours) if hours else None)


def scope_reasons_demographic(patient: dict) -> list[str]:
    r = []
    if patient["age"] < 18:
        r.append("This tool is for adults (18+) only. Children's doses depend on weight and product - "
                 "please ask a pharmacist or doctor.")
    if patient.get("pregnant"):
        r.append("Medicines in pregnancy need professional advice.")
    if patient.get("breastfeeding"):
        r.append("Medicines while breastfeeding need professional advice.")
    return r


def scope_reasons_history(patient: dict) -> list[str]:
    r = []
    conds = set(patient.get("conditions") or [])
    if "kidney_disease" in conds:
        r.append("Long-term kidney disease changes which medicines and doses are safe.")
    if "liver_disease" in conds:
        r.append("Long-term liver disease changes which medicines and doses are safe.")
    if patient["age"] >= 75 and (patient.get("num_regular_meds") or 0) >= 2:
        r.append("At 75+ with several regular medicines, a pharmacist should review interactions.")
    return r




### Cell 2.4 — Start of the `evaluate()` function
`evaluate(patient)` is the single entry point of the engine.
* `_new_result` creates an empty answer with all fields present.
* `_finish` fills in the outcome (`action`), which rule produced it, and any extra details.

The next six cells continue **the same function**, one rule per cell, in the exact order they run.

In [ ]:
%%writefile -a rules.py
# --------------------------------------------------------------------------
# Engine
# --------------------------------------------------------------------------
def _new_result(kb: Knowledge) -> dict:
    return {"action": None, "rule": None, "reasons": [], "missing": [],
            "triggered_flags": [], "options": [], "excluded": [],
            "stop_advice": None, "disclaimer": DISCLAIMER,
            "emergency_number": kb.emergency_number}


def _finish(res: dict, action: str, rule: int, **kw) -> dict:
    res["action"] = action
    res["rule"] = rule
    for k, v in kw.items():
        res[k] = v
    return res


def evaluate(patient: dict, kb: Knowledge | None = None) -> dict:
    """Run Rules 1-6 in order; return a result dict."""
    kb = kb or default_knowledge()
    res = _new_result(kb)



### Cell 2.5 — **Rule 1: red-flag triage** (always first)
1. If the complaint is unknown → `ASK` for it.
2. If **any** red flag was answered "Yes" → stop immediately with `EMERGENCY` (if any flag has emergency severity) or `URGENT_DOCTOR`.
   No medicine is ever suggested. This happens **even if other questions are still unanswered**, because one danger sign is enough.
3. If any red flag is still unanswered, or the number of days is missing → `ASK`.

In [ ]:
%%writefile -a rules.py
    # ---- Rule 1: red-flag triage -------------------------------------------
    complaint = patient.get("complaint")
    if complaint not in kb.complaints:
        return _finish(res, "ASK", 1, missing=["complaint"])
    flags = kb.red_flags_for(complaint)
    answers = patient.get("red_flags") or {}
    triggered = [f for f in flags if answers.get(f["id"]) is True]
    if triggered:  # any "yes" escalates immediately, even if others unanswered
        action = ("EMERGENCY" if any(f["severity"] == "emergency" for f in triggered)
                  else "URGENT_DOCTOR")
        return _finish(res, action, 1, triggered_flags=triggered,
                       reasons=["Red-flag symptom reported - no medicine is suggested."])
    missing = [f"red_flags.{f['id']}" for f in flags if answers.get(f["id"]) not in (True, False)]
    days = patient.get("symptom_days")
    if not _is_num(days) or days < 0:
        missing.append("symptom_days")
    if missing:
        return _finish(res, "ASK", 1, missing=missing)



### Cell 2.6 — **Rule 2: demographics and scope**
1. Age, weight and sex are required. For women and "other", pregnancy and breastfeeding status are required too. Nothing is assumed: missing → `ASK`.
2. Children, pregnancy and breastfeeding → `REFER`, labelled `scope="out_of_scope"`.

In [ ]:
%%writefile -a rules.py
    # ---- Rule 2: demographics and scope ------------------------------------
    missing = [f for f in ("age", "weight_kg") if not _is_num(patient.get(f))]
    if patient.get("sex") not in SEXES:
        missing.append("sex")
    elif patient["sex"] != "male":
        missing += [f for f in ("pregnant", "breastfeeding")
                    if not isinstance(patient.get(f), bool)]
    if missing:
        return _finish(res, "ASK", 2, missing=missing)
    out = scope_reasons_demographic(patient)
    if out:
        return _finish(res, "REFER", 2, reasons=out, scope="out_of_scope")



### Cell 2.7 — **Rule 3: contraindications**
1. Health conditions, regular medicines, allergies, alcohol use and the number of regular medicines must be answered.
2. Kidney disease, liver disease, or age 75+ with several medicines → `REFER` (out of scope).
3. Each candidate medicine is checked. Unsafe ones go into `excluded`, with the reason shown to the user.
4. If **no** medicine is left → `REFER` to a pharmacist or doctor.

In [ ]:
%%writefile -a rules.py
    # ---- Rule 3: contraindications -----------------------------------------
    missing = [f for f in HISTORY_FIELDS if patient.get(f) is None]
    if missing:
        return _finish(res, "ASK", 3, missing=missing)
    out = scope_reasons_history(patient)
    if out:
        return _finish(res, "REFER", 3, reasons=out, scope="out_of_scope")
    eligible = []
    for key in kb.candidates_for(complaint):
        drug = kb.drugs[key]
        why = contraindication_reasons(drug, patient)
        if why:
            res["excluded"].append({"drug": key, "display_name": drug["display_name"], "reasons": why})
        else:
            eligible.append(key)
    if not eligible:
        return _finish(res, "REFER", 3, reasons=[
            "None of the first-line medicines for this problem are suitable for you. "
            "A pharmacist or doctor can suggest a safe alternative."])



### Cell 2.8 — **Rule 4: the 24-hour cap**
1. The engine must know what was already taken in the last 24 hours. If a dose was taken, the time since that dose is required.
2. For each remaining medicine: `remaining = personal daily maximum − amount already taken`.
3. If even the smallest dose would break the cap → `REFER` with *"Do NOT take any more"*. This is the conservative design: the engine refers rather than switching to another drug.
4. If the weight-adjusted dose is smaller than one tablet, that medicine is excluded.

In [ ]:
%%writefile -a rules.py
    # ---- Rule 4: 24-hour cap and interval ----------------------------------
    intake = patient.get("prior_intake")
    if intake is None:
        return _finish(res, "ASK", 4, missing=["prior_intake"])
    if any((e.get("total_mg_24h") or 0) > 0 and not _is_num(e.get("hours_since_last"))
           for e in intake):
        return _finish(res, "ASK", 4, missing=["prior_intake"])
    plans = []
    for key in eligible:
        drug = kb.drugs[key]
        lim = effective_limits(drug, patient)
        if lim["single_max"] < lim["single_min"]:
            res["excluded"].append({"drug": key, "display_name": drug["display_name"],
                                    "reasons": ["Body weight too low for the standard tablet strength"]})
            continue
        prior, hrs = prior_intake_for(patient, drug["ingredient"])
        remaining = lim["max_24h"] - prior
        if remaining < lim["single_min"]:
            return _finish(res, "REFER", 4, scope="cap_reached", reasons=[
                f"You have already taken {prior:g} mg of {drug['display_name']} in the last 24 hours "
                f"(maximum for you: {lim['max_24h']} mg). Do NOT take any more. "
                "Speak to a pharmacist; if you took more than the maximum, get medical help now."])
        plans.append((key, drug, lim, prior, hrs, remaining))
    if not plans:
        return _finish(res, "REFER", 4, reasons=["No standard dose is suitable - ask a pharmacist."])



### Cell 2.9 — **Rule 5: self-care duration**
If the patient has had the symptom longer than `max_self_care_days` (for example, headache > 3 days, diarrhoea > 2 days) → `REFER` to a doctor.

In [ ]:
%%writefile -a rules.py
    # ---- Rule 5: self-care duration limit ----------------------------------
    stop_advice = kb.complaints[complaint]["stop_advice"]
    max_days = kb.max_self_care_days(complaint)
    if days > max_days:
        return _finish(res, "REFER", 5, stop_advice=stop_advice, reasons=[
            f"You have had this for {days:g} days; self-treatment is only advised for up to "
            f"{max_days} days. Please see a doctor."])



### Cell 2.10 — **Rule 6: build the recommendation**
Every safe medicine becomes an **option card** containing: drug name, strength, dose range, how often, personal 24-h maximum,
amount already taken, remaining allowance, hours to wait before the next dose (`next_dose_in_h`), warnings, interactions,
source and verification status.

* `_build_option` caps the dose by `min(single-dose maximum, remaining allowance)`, rounded down to whole tablets.
* Loperamide uses its special pattern: 4 mg first, then 2 mg.
* `_count` turns mg into a tablet count for display, for example "1-2 tablets".

In [ ]:
%%writefile -a rules.py
    # ---- Rule 6: build the recommendation ----------------------------------
    options = [_build_option(key, drug, lim, prior, hrs, remaining)
               for key, drug, lim, prior, hrs, remaining in plans]
    return _finish(res, "RECOMMEND", 6, options=options, stop_advice=stop_advice)


def _count(mg: int, strength: int, unit: str) -> str:
    n = mg // strength
    return f"{n} {unit}{'s' if n != 1 else ''}"


def _build_option(key, drug, lim, prior, hrs, remaining) -> dict:
    s = drug["strength_mg"]
    cap_now = floor_to(remaining, s)
    if "first_dose_mg" in drug:  # e.g., loperamide: 4 mg first, then 2 mg
        dose = drug["first_dose_mg"] if prior == 0 else drug["subsequent_dose_mg"]
        dose = min(dose, lim["single_max"], cap_now)
        dmin = dmax = dose
    else:
        dmin = lim["single_min"]
        dmax = min(lim["single_max"], cap_now)
    unit = drug["unit"]
    if dmin == dmax:
        dose_text = f"{dmin} mg ({_count(dmin, s, unit)})"
    else:
        dose_text = f"{dmin}-{dmax} mg ({dmin // s}-{dmax // s} {unit}s)"
    next_in = 0.0
    if prior > 0 and hrs is not None:
        next_in = max(0.0, drug["min_interval_h"] - hrs)
    return {
        "drug_key": key,
        "display_name": drug["display_name"],
        "strength_mg": s,
        "form": drug["form"],
        "dose_min_mg": dmin,
        "dose_max_mg": dmax,
        "dose_text": dose_text,
        "interval_text": drug["interval_text"],
        "min_interval_h": drug["min_interval_h"],
        "max_24h_mg": lim["max_24h"],
        "prior_mg_24h": prior,
        "remaining_mg_24h": lim["max_24h"] - prior,
        "next_dose_in_h": next_in,
        "administration": drug["administration"],
        "warnings": list(drug["warnings"]),
        "interactions": list(drug.get("interactions", [])),
        "limit_notes": lim["notes"],
        "source": drug["source"],
        "verification_status": drug["status"],
    }


---
# Part 3 — The safety validator and red alarm

### Cell 3.1 — `validator.py`, part 1: shared checks
The validator is a **second, independent safety net**. It deliberately **re-implements the dose arithmetic itself**
(`patient_limits`, `_prior`) instead of calling the engine's functions. A bug in `rules.py` is therefore not silently repeated here.
This is the same idea as a second pharmacist double-checking a prescription.

`validate(patient, result)` returns a list of failures. **Empty list = safe to show.** Checks in this part:
* **V0:** the action is one of the five allowed values.
* **V1:** if any red flag is "Yes", the answer **must** be an escalation.
* **V2:** a referral or emergency answer must not carry medicine options.

In [ ]:
%%writefile validator.py
"""
validator.py - Independent safety validator (defence in depth).

Runs AFTER the rules engine and BEFORE anything is shown to the user.
It deliberately re-implements the safety arithmetic instead of calling the
engine's helper functions, so that a bug in rules.py is not silently
repeated here. It only reads data (formulary/triage) from the Knowledge object.

Checks
  V0  action is a known value
  V1  red flags: any "yes" must be escalated; a recommendation needs a
      complete and fully negative triage
  V2  no options attached to a non-recommendation
  V3  patient in scope (adult, not pregnant/breastfeeding, no kidney/liver
      disease, not 75+ with >= 2 regular medicines)
  V4  symptom duration within the self-care limit
  V5  every option has all required display fields
  V6  drug exists in the formulary, is a candidate for this complaint, and
      its name/strength match the formulary (no invented drugs)
  V7  dose sanity: 0 < min <= max, whole tablets
  V8  dose <= single-dose maximum (patient-specific)
  V9  prior intake + dose <= 24-hour maximum (patient-specific)
  V10 minimum interval respected
  V11 no contraindicated drug (condition, interacting medicine, allergy, age)
  V12 displayed 24-h maximum and remaining allowance are correct
  V13 disclaimer present
"""
from __future__ import annotations

from rules import ACTIONS, DISCLAIMER, Knowledge, default_knowledge

ESCALATE = {"EMERGENCY", "URGENT_DOCTOR"}
REQUIRED_OPTION_FIELDS = (
    "drug_key", "display_name", "strength_mg", "form", "dose_min_mg", "dose_max_mg",
    "dose_text", "interval_text", "min_interval_h", "max_24h_mg", "prior_mg_24h",
    "remaining_mg_24h", "next_dose_in_h", "warnings", "source", "verification_status",
)


def _num(x) -> bool:
    return isinstance(x, (int, float)) and not isinstance(x, bool)


def patient_limits(drug: dict, p: dict) -> tuple[float, float]:
    """Independent computation of (single-dose max, 24-h max) in mg."""
    s = drug["strength_mg"]
    single_max = drug["single_dose_max_mg"]
    max24 = drug["max_24h_mg"]
    red = drug.get("reduced_limits") or {}
    if red:
        w = p.get("weight_kg")
        applies = red.get("applies_if", [])
        low = _num(w) and w < red.get("low_weight_threshold_kg", 0)
        if (p.get("alcohol_regular") and "alcohol_regular" in applies) or (low and "low_weight" in applies):
            max24 = min(max24, red["max_24h_mg"])
        if low:
            single_max = min(single_max, (red["per_kg_single_mg"] * w + 1e-9) // s * s)
            max24 = min(max24, (red["per_kg_24h_mg"] * w + 1e-9) // s * s)
    return single_max, max24


def _prior(p: dict, ingredient: str) -> tuple[float, float | None]:
    total, last = 0.0, None
    for e in p.get("prior_intake") or []:
        if e.get("ingredient") != ingredient:
            continue
        mg = e.get("total_mg_24h") or 0
        if mg > 0:
            total += mg
            h = e.get("hours_since_last")
            if _num(h):
                last = h if last is None else min(last, h)
    return total, last


def validate(patient: dict, result: dict, kb: Knowledge | None = None) -> list[dict]:
    """Return a list of failures; an empty list means the output is safe to show."""
    kb = kb or default_knowledge()
    fails: list[dict] = []

    def fail(check: str, detail: str):
        fails.append({"check": check, "detail": detail})

    action = result.get("action")
    if action not in ACTIONS:
        fail("V0_action", f"Unknown action {action!r}")
        return fails

    complaint = patient.get("complaint")
    answers = patient.get("red_flags") or {}
    flags = kb.red_flags_for(complaint) if complaint in kb.complaints else []
    positive = [f["id"] for f in flags if answers.get(f["id"]) is True]
    if positive and action not in ESCALATE:
        fail("V1_red_flag", f"Red flag(s) {positive} reported but action is {action}")

    if action != "RECOMMEND":
        if result.get("options"):
            fail("V2_options_on_non_recommend", "Medicine options attached to a non-recommendation")
        return fails



### Cell 3.2 — `validator.py`, part 2: patient-level checks for a recommendation
* **V1:** triage complete and **all** red flags answered "No".
* **V3:** patient in scope (adult; pregnancy and breastfeeding answered "No"; no kidney or liver disease; not 75+ with several medicines).
* **V4:** symptom duration within the self-care limit.
* **V13:** the mandatory disclaimer is present and unchanged.

In [ ]:
%%writefile -a validator.py
    # ---- patient-level checks for a recommendation -------------------------
    if complaint not in kb.complaints:
        fail("V1_red_flag", "Recommendation without a known complaint / triage")
        return fails
    incomplete = [f["id"] for f in flags if answers.get(f["id"]) is not False]
    if incomplete:
        fail("V1_red_flag", f"Triage not completed/negative for: {incomplete}")

    age = patient.get("age")
    conds = set(patient.get("conditions") or [])
    if not _num(age) or age < 18:
        fail("V3_scope", "Patient is not a confirmed adult")
    if patient.get("pregnant") is True or patient.get("breastfeeding") is True:
        fail("V3_scope", "Pregnant or breastfeeding")
    if patient.get("sex") != "male" and (patient.get("pregnant") is None or patient.get("breastfeeding") is None):
        fail("V3_scope", "Pregnancy/breastfeeding status unknown")
    if conds & {"kidney_disease", "liver_disease"}:
        fail("V3_scope", "Chronic kidney/liver disease")
    if _num(age) and age >= 75 and (patient.get("num_regular_meds") or 0) >= 2:
        fail("V3_scope", "Age 75+ with multiple regular medicines")

    days = patient.get("symptom_days")
    if not _num(days) or days > kb.max_self_care_days(complaint):
        fail("V4_duration", f"Symptom duration {days} exceeds self-care limit")

    if result.get("disclaimer") != DISCLAIMER:
        fail("V13_disclaimer", "Mandatory disclaimer missing or altered")

    options = result.get("options") or []
    if not options:
        fail("V5_fields", "Recommendation contains no options")

    meds = set(patient.get("current_meds") or [])
    allergies = set(patient.get("allergies") or [])


### Cell 3.3 — `validator.py`, part 3: checks on every medicine option
| Check | Catches |
|---|---|
| **V5** | A required display field is missing (for example, warnings) |
| **V6** | A drug **not in the formulary**, a drug not listed for this complaint, or a wrong name or strength (invented drugs) |
| **V7** | Nonsense doses: zero, min > max, or not whole tablets |
| **V8** | Dose above the patient's **single-dose maximum** |
| **V9** | Earlier intake + this dose above the **24-hour maximum** |
| **V10** | Next dose suggested **too soon** after the last one |
| **V11** | Drug **contraindicated** by a condition, medicine, allergy or age |
| **V12** | The card shows a wrong daily maximum or remaining allowance |

In [ ]:
%%writefile -a validator.py
    for opt in options:
        miss = [f for f in REQUIRED_OPTION_FIELDS if opt.get(f) in (None, "", [])]
        if miss:
            fail("V5_fields", f"{opt.get('drug_key')}: missing {miss}")
            continue
        key = opt["drug_key"]
        drug = kb.drugs.get(key)
        if drug is None:
            fail("V6_formulary", f"Drug {key!r} is not in the formulary")
            continue
        if key not in kb.complaints[complaint]["candidates"] or complaint not in drug["indications"]:
            fail("V6_formulary", f"{key} is not a listed option for {complaint}")
        if opt["display_name"] != drug["display_name"] or opt["strength_mg"] != drug["strength_mg"]:
            fail("V6_formulary", f"{key}: name/strength differ from the formulary")

        s = drug["strength_mg"]
        dmin, dmax = opt["dose_min_mg"], opt["dose_max_mg"]
        if not (_num(dmin) and _num(dmax) and 0 < dmin <= dmax) or dmin % s or dmax % s:
            fail("V7_dose_sanity", f"{key}: invalid dose range {dmin}-{dmax} mg")
            continue

        single_max, max24 = patient_limits(drug, patient)
        if dmax > single_max:
            fail("V8_single_dose", f"{key}: {dmax} mg exceeds single-dose max {single_max:g} mg")
        prior, last = _prior(patient, drug["ingredient"])
        if prior + dmax > max24:
            fail("V9_24h_cap", f"{key}: {prior:g} + {dmax} mg exceeds 24-h max {max24:g} mg")
        if prior > 0:
            if last is None:
                fail("V10_interval", f"{key}: time of last dose unknown")
            elif last < drug["min_interval_h"] and opt["next_dose_in_h"] + 1e-6 < drug["min_interval_h"] - last:
                fail("V10_interval", f"{key}: minimum interval of {drug['min_interval_h']} h not respected")

        ci = drug.get("contraindications") or {}
        bad = (conds & set(ci.get("conditions", []))) | (meds & set(ci.get("med_categories", []))) \
            | (allergies & set(ci.get("allergies", [])))
        if bad:
            fail("V11_contraindication", f"{key}: contraindicated by {sorted(bad)}")
        if drug.get("avoid_if_age_gte") is not None and _num(age) and age >= drug["avoid_if_age_gte"]:
            fail("V11_contraindication", f"{key}: not for age {drug['avoid_if_age_gte']}+")

        if opt["max_24h_mg"] != max24 or abs(opt["remaining_mg_24h"] - (max24 - prior)) > 1e-6:
            fail("V12_display", f"{key}: displayed cap/remaining allowance is wrong")
    return fails


### Cell 3.4 — `incident_log.py`: the incident record
Appends one row to `incidents.csv` whenever (a) the validator blocks an answer, or (b) a user presses **"Report incorrect advice"**.
Each row stores: UTC time, kind of incident, failed checks, the patient's structured answers (no names), the blocked output, the
user's comment and the app version. These rows are what a clinician reviews to find and fix errors.

In [ ]:
%%writefile incident_log.py
"""
incident_log.py - Append-only CSV log for validator blocks and user reports.

Only the structured, non-identifying intake fields are stored (no names,
phone numbers or free-text identity data are collected by the app).
Set MEDBOT_INCIDENT_LOG to change the file location.
"""
from __future__ import annotations

import csv
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from rules import APP_VERSION, BASE_DIR

FIELDS = ["timestamp_utc", "kind", "failed_checks", "inputs_json",
          "proposed_output_json", "comment", "app_version"]


def log_path(path: str | os.PathLike | None = None) -> Path:
    return Path(path or os.environ.get("MEDBOT_INCIDENT_LOG") or BASE_DIR / "incidents.csv")


def log_incident(kind: str, patient: dict, proposed: dict, failures: list | None = None,
                 comment: str = "", path: str | os.PathLike | None = None) -> Path:
    """kind: 'validator_block' or 'user_report'."""
    p = log_path(path)
    new = not p.exists()
    p.parent.mkdir(parents=True, exist_ok=True)
    row = {
        "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "kind": kind,
        "failed_checks": "; ".join(f"{f['check']}: {f['detail']}" for f in failures or []),
        "inputs_json": json.dumps(patient, sort_keys=True, default=str),
        "proposed_output_json": json.dumps(proposed, sort_keys=True, default=str),
        "comment": (comment or "")[:2000],
        "app_version": APP_VERSION,
    }
    with p.open("a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=FIELDS)
        if new:
            w.writeheader()
        w.writerow(row)
    return p


### Cell 3.5 — `pipeline.py`: the only route from answers to the screen
```
rules.evaluate()  →  validator.validate()  →  answer shown,  OR  BLOCKED + red alarm + incident log
```
* If the validator finds **any** failure, the original output is replaced by a `BLOCKED` result that contains **no medicines**,
  only the message *"Safety check failed — recommendation withheld"* and a safe fallback (see a pharmacist or doctor).
* `engine=` can be swapped out. The evaluation (Part 8) uses this to feed the validator **deliberately broken outputs**
  and prove it catches them.

In [ ]:
%%writefile pipeline.py
"""
pipeline.py - The only path from patient input to a displayable answer:

    rules.evaluate()  ->  validator.validate()  ->  result or RED-ALARM block
"""
from __future__ import annotations

from incident_log import log_incident
from rules import DISCLAIMER, Knowledge, default_knowledge, evaluate
from validator import validate

BLOCK_MESSAGE = "Safety check failed - recommendation withheld"
FALLBACK = ("Please speak to a pharmacist or doctor before taking any medicine. "
            "If you feel very unwell, call the emergency number.")


def run_pipeline(patient: dict, kb: Knowledge | None = None,
                 log: bool = False, log_file=None, engine=evaluate) -> dict:
    """Evaluate, validate and (if any check fails) block the output.

    `engine` is injectable so the evaluation harness can test the validator
    against deliberately faulty engine outputs.
    """
    kb = kb or default_knowledge()
    proposed = engine(patient, kb)
    failures = validate(patient, proposed, kb)
    if failures:
        if log:
            log_incident("validator_block", patient, proposed, failures, path=log_file)
        return {
            "action": "BLOCKED", "rule": "validator", "reasons": [BLOCK_MESSAGE, FALLBACK],
            "failures": failures, "proposed": proposed, "options": [], "missing": [],
            "triggered_flags": [], "excluded": [], "stop_advice": None,
            "disclaimer": DISCLAIMER, "emergency_number": kb.emergency_number,
        }
    proposed["validator"] = "passed"
    return proposed


---
# Part 4 — Try the engine on example patients (optional demo)

### Cell 4.1 — Six example patients, one per type of outcome
This cell does not write a file. It **runs the engine directly** so you can see each type of decision without opening the app:

1. Healthy adult with a headache → **RECOMMEND**
2. Headache with a "worst-ever, sudden" red flag → **EMERGENCY**
3. Already took 4000 mg paracetamol in cold & flu products → **REFER** (cap reached)
4. 16-year-old → **REFER** (out of scope)
5. Back pain + stomach-ulcer history → **RECOMMEND paracetamol only** (ibuprofen excluded)
6. Weight 45 kg → paracetamol dose **reduced** to 500 mg, daily cap 2500 mg

`make_patient()` builds a complete low-risk adult; each example changes only what it needs.

In [ ]:
import importlib, sys
sys.path.insert(0, ".")
import rules, validator, pipeline                       # the files we just wrote
for m in (rules, validator, pipeline):
    importlib.reload(m)                                 # pick up edits if you re-run the notebook

kb = rules.default_knowledge()

def make_patient(complaint="headache", **changes):
    """A complete, low-risk adult patient. Pass keyword arguments to change any answer."""
    p = dict(complaint=complaint, symptom_days=1,
             red_flags={f["id"]: False for f in kb.red_flags_for(complaint)},   # all red flags = "No"
             age=30, sex="female", weight_kg=65, pregnant=False, breastfeeding=False,
             conditions=[], current_meds=[], allergies=[], alcohol_regular=False,
             num_regular_meds=0, prior_intake=[])
    p.update(changes)
    return p

def show(title, patient):
    """Run the full pipeline (engine + validator) and print a one-line summary."""
    r = pipeline.run_pipeline(patient, kb)
    detail = "; ".join(f"{o['display_name']} {o['dose_text']} (max {o['max_24h_mg']} mg/24h)"
                       for o in r["options"]) or " | ".join(r["reasons"])
    print(f"{title:<45} → {r['action']:<14} {detail}")

red = make_patient(); red["red_flags"]["thunderclap"] = True
show("1. Healthy adult, headache", make_patient())
show("2. Sudden 'worst-ever' headache", red)
show("3. 4000 mg paracetamol already taken",
     make_patient(prior_intake=[{"ingredient": "paracetamol", "total_mg_24h": 4000, "hours_since_last": 3}]))
show("4. Age 16", make_patient(age=16))
show("5. Back pain + stomach ulcer history",
     make_patient("musculoskeletal_pain", conditions=["peptic_ulcer_or_gi_bleed"]))
show("6. Weight 45 kg", make_patient(weight_kg=45))

### Cell 4.2 — Watch the red alarm fire
Here we **simulate a bug**: a fake engine that triples the dose. The validator catches it, and the pipeline returns
`BLOCKED`, lists the failed checks, and contains no medicine. `log=False` keeps this demo out of the real incident file.

In [ ]:
def buggy_engine(patient, kb):
    result = rules.evaluate(patient, kb)
    for option in result["options"]:
        option["dose_max_mg"] *= 3          # deliberate overdose bug
    return result

blocked = pipeline.run_pipeline(make_patient(), kb, engine=buggy_engine, log=False)
print("Action:", blocked["action"])
print("Message:", blocked["reasons"][0])
print("Failed checks:")
for f in blocked["failures"]:
    print("  -", f["check"], "→", f["detail"])

---
# Part 5 — Optional AI intake and bilingual text

### Cell 5.1 — `llm_intake.py`: optional free-text pre-fill
If you add an Anthropic API key, a user can type a description such as *"I'm 34, 60 kg, headache since yesterday, took 2 flu tablets"*
and the form is **pre-filled**. Safety design:
* The AI is asked only to **extract** fields. Its prompt forbids suggesting medicines.
* `sanitize()` keeps **only whitelisted fields with valid types and ranges**. Anything else, such as an invented "dose" field, is thrown away.
* The AI may pre-mark a red flag as **"Yes"** (safer) but **never "No"**.
* The user must still confirm every answer in the form. Without a key, this file is simply not used.

In [ ]:
%%writefile llm_intake.py
"""
llm_intake.py - OPTIONAL free-text intake using an LLM.

The LLM only extracts structured fields from the user's description to
PRE-FILL the guided form. It never produces drug names or doses, and its
output never reaches the rules engine directly: the user must confirm every
answer in the form. Safety properties:
  * strict whitelist sanitisation of every field (unknown keys are dropped);
  * red flags can only be pre-marked "Yes" (escalation), never "No";
  * the app works fully without an API key.

Provider: Anthropic Messages API over HTTPS (set ANTHROPIC_API_KEY, and
optionally MEDBOT_LLM_MODEL).
"""
from __future__ import annotations

import json
import os
import re

import requests

from rules import ALLERGY_OPTIONS, CONDITION_OPTIONS, MED_CATEGORY_OPTIONS, SEXES, Knowledge

API_URL = "https://api.anthropic.com/v1/messages"
DEFAULT_MODEL = "claude-haiku-5-5"


class IntakeError(RuntimeError):
    pass


def _system_prompt(kb: Knowledge) -> str:
    flag_ids = sorted({f["id"] for c in kb.complaints for f in kb.red_flags_for(c)})
    ingredients = sorted({d["ingredient"] for d in kb.drugs.values()})
    return (
        "You extract structured intake data from a patient's description for a medicine-safety form.\n"
        "Rules: NEVER suggest, name or dose any medicine of your own. Do not give advice. "
        "Only report what the text explicitly states; use null when unsure.\n"
        "Return ONE JSON object and nothing else, with keys:\n"
        f"  complaint: one of {sorted(kb.complaints)} or null\n"
        "  symptom_days: number or null\n  age: number or null\n"
        f"  sex: one of {list(SEXES)} or null\n  weight_kg: number or null\n"
        "  pregnant, breastfeeding, alcohol_regular: true/false/null\n"
        f"  conditions: subset of {sorted(CONDITION_OPTIONS)}\n"
        f"  current_meds: subset of {sorted(MED_CATEGORY_OPTIONS)}\n"
        f"  allergies: subset of {sorted(ALLERGY_OPTIONS)}\n"
        "  num_regular_meds: number or null\n"
        f"  prior_intake: list of {{ingredient (one of {ingredients}), total_mg_24h, hours_since_last}}\n"
        f"  red_flags_positive: subset of {flag_ids} - ONLY flags the text clearly describes.\n"
    )


def _as_num(v, lo, hi):
    if isinstance(v, (int, float)) and not isinstance(v, bool) and lo <= v <= hi:
        return v
    return None


def sanitize(raw: dict, kb: Knowledge) -> dict:
    """Keep only whitelisted, type-checked fields."""
    out: dict = {}
    if raw.get("complaint") in kb.complaints:
        out["complaint"] = raw["complaint"]
    for key, lo, hi in (("symptom_days", 0, 365), ("age", 0, 120),
                        ("weight_kg", 1, 300), ("num_regular_meds", 0, 50)):
        v = _as_num(raw.get(key), lo, hi)
        if v is not None:
            out[key] = v
    if raw.get("sex") in SEXES:
        out["sex"] = raw["sex"]
    for key in ("pregnant", "breastfeeding", "alcohol_regular"):
        if isinstance(raw.get(key), bool):
            out[key] = raw[key]
    for key, allowed in (("conditions", CONDITION_OPTIONS), ("current_meds", MED_CATEGORY_OPTIONS),
                         ("allergies", ALLERGY_OPTIONS)):
        vals = raw.get(key)
        if isinstance(vals, list):
            out[key] = [v for v in vals if v in allowed]
    ingredients = {d["ingredient"] for d in kb.drugs.values()}
    intake = []
    for e in raw.get("prior_intake") or []:
        if isinstance(e, dict) and e.get("ingredient") in ingredients:
            mg = _as_num(e.get("total_mg_24h"), 0, 20000)
            if mg is not None:
                intake.append({"ingredient": e["ingredient"], "total_mg_24h": mg,
                               "hours_since_last": _as_num(e.get("hours_since_last"), 0, 24)})
    if intake:
        out["prior_intake"] = intake
    all_flags = {f["id"] for c in kb.complaints for f in kb.red_flags_for(c)}
    pos = raw.get("red_flags_positive")
    if isinstance(pos, list):
        out["red_flags_positive"] = [f for f in pos if f in all_flags]
    return out


def extract_fields(text: str, api_key: str, kb: Knowledge, model: str | None = None,
                   timeout: int = 30) -> dict:
    if not api_key:
        raise IntakeError("No API key configured.")
    payload = {
        "model": model or os.environ.get("MEDBOT_LLM_MODEL", DEFAULT_MODEL),
        "max_tokens": 800,
        "system": _system_prompt(kb),
        "messages": [{"role": "user", "content": text[:4000]}],
    }
    headers = {"x-api-key": api_key, "anthropic-version": "2023-06-01",
               "content-type": "application/json"}
    try:
        r = requests.post(API_URL, json=payload, headers=headers, timeout=timeout)
        r.raise_for_status()
        content = "".join(b.get("text", "") for b in r.json().get("content", []))
    except (requests.RequestException, ValueError) as exc:
        raise IntakeError(f"AI pre-fill unavailable: {exc}") from exc
    match = re.search(r"\{.*\}", content, re.S)
    if not match:
        raise IntakeError("AI pre-fill returned no usable data.")
    try:
        raw = json.loads(match.group(0))
    except json.JSONDecodeError as exc:
        raise IntakeError("AI pre-fill returned malformed data.") from exc
    return sanitize(raw, kb)


### Cell 5.2 — `i18n.py`: English / Urdu interface text
Holds the interface wording (titles, emergency screen, buttons) in English and Urdu. `t(key, lang)` returns the right string
and falls back to English. The **medical questions stay in English** until a professional clinical translator reviews them,
because a mistranslated danger sign is a safety risk.

In [ ]:
%%writefile i18n.py
"""
i18n.py - English / Urdu strings for the interface chrome.

Clinical questions (red flags, contraindications) stay in English in v1:
translating them without professional clinical-translation review could
change their meaning, which is a safety risk.
"""

STRINGS = {
    "en": {
        "app_title": "OTC Safe-Dose Assistant",
        "app_subtitle": "Safety-first guidance on over-the-counter medicines for common adult complaints",
        "disclaimer": "This is general information, not a diagnosis. Confirm with a pharmacist or doctor. "
                      "In an emergency call",
        "step_triage": "Triage",
        "step_profile": "Profile",
        "step_safety": "Safety checks",
        "step_rec": "Recommendation",
        "emergency_title": "EMERGENCY - get medical help now",
        "emergency_body": "Your answers suggest you may need urgent medical attention. No medicine is suggested.",
        "call_now": "Call now, or go to the nearest emergency department",
        "urgent_title": "See a doctor today",
        "urgent_body": "Your symptoms need a doctor's assessment. Please do not self-treat. "
                       "If things get worse, call the emergency number.",
        "refer_title": "Please speak to a pharmacist or doctor",
        "alarm": "Safety check failed - recommendation withheld",
        "safety_ok": "All safety checks passed",
        "start_over": "Start over",
        "language": "Language",
        "profile": "Patient profile",
        "clinical_en_only": "",
        "privacy": "Privacy: answers stay in this browser session and are not stored. "
                   "Only safety incidents and reports you choose to send are logged, without names.",
    },
    "ur": {
        "app_title": "او ٹی سی محفوظ خوراک معاون",
        "app_subtitle": "عام بیماریوں کے لیے بغیر نسخے والی دواؤں کی محفوظ رہنمائی — صرف بالغ افراد کے لیے",
        "disclaimer": "یہ عمومی معلومات ہیں، تشخیص نہیں۔ کسی فارماسسٹ یا ڈاکٹر سے تصدیق کریں۔ ایمرجنسی میں کال کریں",
        "step_triage": "ابتدائی جانچ",
        "step_profile": "مریض کی معلومات",
        "step_safety": "حفاظتی جانچ",
        "step_rec": "سفارش",
        "emergency_title": "ہنگامی صورتحال — فوراً طبی مدد حاصل کریں",
        "emergency_body": "آپ کی بتائی گئی علامات کو فوری طبی توجہ کی ضرورت ہو سکتی ہے۔ کوئی دوا تجویز نہیں کی جا رہی۔",
        "call_now": "ابھی اس نمبر پر کال کریں یا قریبی ایمرجنسی میں جائیں",
        "urgent_title": "آج ہی ڈاکٹر سے ملیں",
        "urgent_body": "آپ کی علامات کا ڈاکٹر سے معائنہ ضروری ہے۔ خود سے علاج نہ کریں۔ حالت بگڑنے پر ایمرجنسی نمبر پر کال کریں۔",
        "refer_title": "فارماسسٹ یا ڈاکٹر سے مشورہ کریں",
        "alarm": "حفاظتی جانچ ناکام — سفارش روک دی گئی",
        "safety_ok": "تمام حفاظتی جانچیں کامیاب",
        "start_over": "دوبارہ شروع کریں",
        "language": "زبان",
        "profile": "مریض کی معلومات",
        "clinical_en_only": "طبی سوالات فی الحال صرف انگریزی میں ہیں؛ ان کا ترجمہ ماہرین کی تصدیق کے بعد شامل کیا جائے گا۔",
        "privacy": "رازداری: آپ کے جوابات محفوظ نہیں کیے جاتے؛ صرف حفاظتی واقعات اور آپ کی بھیجی گئی رپورٹس نام کے بغیر محفوظ ہوتی ہیں۔",
    },
}


def t(key: str, lang: str = "en") -> str:
    return STRINGS.get(lang, STRINGS["en"]).get(key) or STRINGS["en"].get(key, key)


---
# Part 6 — The web interface (`app.py`, Streamlit)

**How Streamlit works:** the script runs **top to bottom on every click**. Values that must survive between clicks
are kept in `st.session_state` (here, `ss.patient` holds the answers). The interface is split over 7 cells.

### Cell 6.1 — Imports and session state
* `st.set_page_config` sets the browser-tab title, icon and wide layout.
* `get_kb()` loads the knowledge base once (`@st.cache_resource`).
* `ss` (session state) stores: `patient` (answers), `lang` (en/ur), `prefill` (AI suggestions), `run_id`
  (changing it resets every input box on "Start over"), `logged` (prevents logging the same incident twice), `api_key`.
* `T()` translates, `k()` makes widget keys, `esc()` makes text safe to put in HTML.

In [ ]:
%%writefile app.py
"""
app.py - Streamlit GUI for the OTC Safe-Dose Assistant.

Flow:  Triage -> Profile -> Safety checks -> Recommendation
Every screen is driven by the rules engine's output (run through the safety
validator): the engine says which answers are still missing, and the GUI
renders the matching step. Nothing is shown that has not passed validation.

Run locally:   streamlit run app.py
"""
from __future__ import annotations

import html
import json
import os

import streamlit as st

import llm_intake
from i18n import t
from incident_log import log_incident
from pipeline import run_pipeline
from rules import (ALLERGY_OPTIONS, APP_VERSION, CONDITION_OPTIONS, MED_CATEGORY_OPTIONS,
                   Knowledge, stage_of_field)

st.set_page_config(page_title="OTC Safe-Dose Assistant", page_icon="💊",
                   layout="wide", initial_sidebar_state="expanded")


@st.cache_resource
def get_kb() -> Knowledge:
    return Knowledge()


KB = get_kb()
ss = st.session_state
for _k, _v in {"patient": {}, "lang": "en", "prefill": {}, "run_id": 0,
               "logged": [], "api_key": ""}.items():
    if _k not in ss:
        ss[_k] = _v


def T(key: str) -> str:
    return t(key, ss.lang)


def k(name: str) -> str:
    """Widget key that resets on 'Start over'."""
    return f"{ss.run_id}_{name}"


def esc(x) -> str:
    return html.escape(str(x))


def rtl() -> str:
    return " rtl" if ss.lang == "ur" else ""




### Cell 6.2 — Styling (CSS)
Custom CSS gives the medical look: a teal-to-blue header, an amber disclaimer bar, a 4-step progress bar, a **full red
emergency box** with a big tap-to-call number, a **pulsing red alarm**, a green "safety passed" badge, recommendation cards,
and the 24-hour limit bar. Colours use transparency so they work in light **and** dark mode, the pulse animation switches off
for users who prefer reduced motion, and the `.rtl` class lays out Urdu right-to-left in a Nastaliq font.

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Styling
# --------------------------------------------------------------------------
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');
.stApp { font-family: 'Inter', system-ui, sans-serif; font-size: 1.04rem; }
.app-header { background: linear-gradient(135deg, #0f766e 0%, #1d4ed8 100%); color: #fff;
  padding: 1.15rem 1.5rem; border-radius: 18px; margin-bottom: .7rem;
  box-shadow: 0 8px 24px rgba(15,118,110,.25); }
.app-header h1 { color: #fff; font-size: 1.75rem; margin: 0; font-weight: 800; padding: 0; }
.app-header p { margin: .25rem 0 0; opacity: .95; }
.disclaimer { border-left: 5px solid #d97706; background: rgba(217,119,6,.10);
  padding: .65rem 1rem; border-radius: 10px; font-size: .97rem; margin-bottom: .4rem; }
.disclaimer a { font-weight: 800; }
.stepper { display: flex; gap: .5rem; margin: .9rem 0 1.1rem; flex-wrap: wrap; }
.step { flex: 1 1 130px; padding: .6rem .7rem; border-radius: 12px; text-align: center;
  border: 1.5px solid rgba(15,118,110,.4); font-weight: 600; font-size: .93rem; opacity: .6; }
.step.done { background: rgba(15,118,110,.15); opacity: 1; }
.step.active { background: #0f766e; color: #fff; opacity: 1; border-color: #0f766e; }
.step.stopped { background: #b91c1c; color: #fff; opacity: 1; border-color: #b91c1c; }
.emergency { background: #b91c1c; color: #fff; padding: 1.8rem 1.4rem; border-radius: 18px; text-align: center; }
.emergency h2 { color: #fff; margin-top: 0; }
.emergency .num a { color: #fff; font-size: 3.4rem; font-weight: 800; text-decoration: none; letter-spacing: .04em; }
.emergency ul { text-align: left; display: inline-block; margin-top: .6rem; }
.urgent { background: #c2410c; color: #fff; padding: 1.5rem 1.3rem; border-radius: 18px; }
.urgent h2 { color: #fff; margin-top: 0; }
.urgent a { color: #fff; font-weight: 800; }
.alarm { background: #b91c1c; color: #fff; padding: 1rem 1.3rem; border-radius: 14px;
  font-weight: 800; font-size: 1.15rem; animation: pulse 1.6s infinite; margin-bottom: .8rem; }
@keyframes pulse { 0% { box-shadow: 0 0 0 0 rgba(185,28,28,.65); }
  70% { box-shadow: 0 0 0 16px rgba(185,28,28,0); } 100% { box-shadow: 0 0 0 0 rgba(185,28,28,0); } }
@media (prefers-reduced-motion: reduce) { .alarm { animation: none; } }
.badge-ok { display: inline-block; background: #15803d; color: #fff; padding: .4rem .95rem;
  border-radius: 999px; font-weight: 700; margin-bottom: .5rem; }
.card { border: 1.5px solid rgba(15,118,110,.4); border-radius: 16px; padding: 1.1rem 1.3rem;
  margin: .7rem 0; background: rgba(15,118,110,.05); }
.card h3 { margin: 0 0 .5rem; padding: 0; }
.card.refer { border-color: rgba(217,119,6,.6); background: rgba(217,119,6,.07); }
table.dose { width: 100%; border-collapse: collapse; margin: .4rem 0 .8rem; }
table.dose th, table.dose td { text-align: left; padding: .45rem .55rem; border-bottom: 1px solid rgba(128,128,128,.25); vertical-align: top; }
table.dose th { width: 38%; font-weight: 600; opacity: .85; }
.capbar { height: 20px; border-radius: 10px; background: rgba(128,128,128,.22); overflow: hidden; display: flex; }
.capbar .prior { background: #64748b; }
.caplabel { font-size: .9rem; margin: .3rem 0 .6rem; }
.wait { background: rgba(217,119,6,.15); border: 1.5px solid #d97706; border-radius: 10px; padding: .55rem .8rem; font-weight: 700; margin-bottom: .6rem; }
.verify { background: rgba(217,119,6,.12); border: 1px dashed #d97706; padding: .45rem .75rem; border-radius: 10px; font-size: .88rem; }
.footer { opacity: .75; font-size: .85rem; margin-top: 2rem; border-top: 1px solid rgba(128,128,128,.25); padding-top: .7rem; }
.rtl { direction: rtl; text-align: right; font-family: 'Noto Nastaliq Urdu', serif; line-height: 2.1; }
</style>
""", unsafe_allow_html=True)




### Cell 6.3 — Header, disclaimer and the sidebar "Patient profile"
* Header with the app name and a **permanent disclaimer** that includes a clickable emergency number.
* Sidebar:
  * English/Urdu switch;
  * a **Patient profile** that fills in as the user answers;
  * a **Start over** button;
  * an optional box for an API key;
  * an About/limitations note.

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Header, sidebar
# --------------------------------------------------------------------------
num = KB.emergency_number
st.markdown(f"""
<div class="app-header{rtl()}"><h1>💊 {esc(T('app_title'))}</h1><p>{esc(T('app_subtitle'))}</p></div>
<div class="disclaimer{rtl()}">⚠️ {esc(T('disclaimer'))} <a href="tel:{esc(num)}">{esc(num)}</a>.</div>
""", unsafe_allow_html=True)

with st.sidebar:
    lang = st.radio(T("language"), ["en", "ur"], horizontal=True,
                    format_func=lambda x: "English" if x == "en" else "اردو",
                    index=0 if ss.lang == "en" else 1)
    if lang != ss.lang:
        ss.lang = lang
        st.rerun()
    st.markdown(f"### 🩺 {T('profile')}")
    p = ss.patient

    def row(label, value):
        if value not in (None, [], {}):
            st.markdown(f"**{label}:** {esc(value)}")

    if p.get("complaint"):
        row("Problem", KB.complaints[p["complaint"]]["label"])
    row("Duration (days)", p.get("symptom_days"))
    if p.get("red_flags"):
        row("Red flags", "Yes ⚠️" if any(p["red_flags"].values()) else "None reported")
    row("Age", p.get("age"))
    row("Sex", (p.get("sex") or "").capitalize() or None)
    row("Weight (kg)", p.get("weight_kg"))
    if p.get("pregnant") or p.get("breastfeeding"):
        row("Pregnant / breastfeeding", "Yes")
    if p.get("conditions") is not None:
        row("Conditions", ", ".join(CONDITION_OPTIONS[c] for c in p["conditions"]) or "None")
    if p.get("current_meds") is not None:
        row("Regular medicines", ", ".join(MED_CATEGORY_OPTIONS[m] for m in p["current_meds"]) or "None")
    if p.get("allergies") is not None:
        row("Allergies", ", ".join(ALLERGY_OPTIONS[a] for a in p["allergies"]) or "None")
    if p.get("prior_intake"):
        row("Taken in last 24 h", "; ".join(f"{e['ingredient']} {e['total_mg_24h']:g} mg"
                                             for e in p["prior_intake"]) or None)
    if not p:
        st.caption("Your answers will appear here as you go.")
    st.divider()
    if st.button(f"🔄 {T('start_over')}", use_container_width=True):
        ss.patient, ss.prefill = {}, {}
        ss.run_id += 1
        st.rerun()
    with st.expander("⚙️ Optional AI text intake"):
        ss.api_key = st.text_input("Anthropic API key", value=ss.api_key, type="password",
                                   help="Optional. Used only to pre-fill the form from a free-text "
                                        "description. Never used to choose medicines or doses.")
        st.caption("Leave empty to use the guided form only (fully functional).")
    with st.expander("ℹ️ About & limitations"):
        st.markdown("Research prototype. **Not a medical device**, not clinically validated, "
                    "no diagnosis. Doses come from a formulary that must be verified by a "
                    "pharmacist. Adults only, OTC medicines only.")




### Cell 6.4 — Run the pipeline and draw the progress stepper
* `run_pipeline(ss.patient)` runs on **every** page load. The interface is entirely **driven by the engine's answer**:
  if the engine says `ASK` for `weight_kg`, the Profile step is shown, and so on.
* If the answer is `BLOCKED`, the incident is logged **once**. The `ss.logged` list prevents duplicates on page refresh.
* `stage_state()` decides which step is active (teal), done (✓), or stopped (red ■) for the stepper at the top.

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Pipeline
# --------------------------------------------------------------------------
result = run_pipeline(ss.patient, KB)
if result["action"] == "BLOCKED":
    sig = json.dumps(ss.patient, sort_keys=True, default=str)
    if sig not in ss.logged:  # log each blocked state once, not on every rerun
        log_incident("validator_block", ss.patient, result["proposed"], result["failures"])
        ss.logged.append(sig)

STAGES = [("triage", "step_triage"), ("profile", "step_profile"),
          ("safety", "step_safety"), ("recommendation", "step_rec")]


def stage_state(res: dict) -> tuple[str, str]:
    a = res["action"]
    if a == "ASK":
        return stage_of_field(res["missing"][0]), "active"
    if a in ("EMERGENCY", "URGENT_DOCTOR"):
        return "triage", "stopped"
    if a == "REFER":
        return {1: "triage", 2: "profile", 3: "safety", 4: "safety"}.get(res["rule"], "recommendation"), "stopped"
    if a == "BLOCKED":
        return "recommendation", "stopped"
    return "recommendation", "done"


cur, state = stage_state(result)
names = [s for s, _ in STAGES]
html_steps = []
for i, (s, label) in enumerate(STAGES):
    cls = "done" if i < names.index(cur) else (state if s == cur else "")
    icon = {"done": "✓", "active": "●", "stopped": "■"}.get(cls, str(i + 1))
    html_steps.append(f'<div class="step {cls}{rtl()}">{icon} {esc(T(label))}</div>')
st.markdown(f'<div class="stepper">{"".join(html_steps)}</div>', unsafe_allow_html=True)




### Cell 6.5 — The three question steps
| Function | Step | What it asks |
|---|---|---|
| `ai_prefill_box` | (optional) | Free-text box. Works only with an API key and only pre-fills answers. |
| `triage_step` | **Triage** | Main problem, number of days, and **every** red-flag question, all Yes/No with **no default** so nothing is assumed. A live red warning appears as soon as any "Yes" is chosen. |
| `profile_step` | **Profile** | Age, weight, sex. Pregnancy and breastfeeding questions appear only when relevant. |
| `safety_step` | **Safety checks** | Conditions, regular medicines, allergies, alcohol, number of medicines, and **all doses taken in the last 24 h**. A hidden-paracetamol warning and a "help me add it up" calculator are included. |

Each **Continue** button refuses to move on until every question is answered, then saves the answers and reruns the page.

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Intake steps
# --------------------------------------------------------------------------
YES_NO = ["No", "Yes"]


def _int(v):
    return int(v) if v is not None else None


def yn_index(v):
    return {False: 0, True: 1}.get(v)


def ai_prefill_box():
    key = ss.api_key or os.environ.get("ANTHROPIC_API_KEY", "")
    with st.container(border=True):
        st.markdown("**💬 Describe your problem in your own words** *(optional)*")
        txt = st.text_area("Description", key=k("free_text"), label_visibility="collapsed",
                           placeholder="e.g., I'm 34, female, about 60 kg. I've had a tension headache "
                                       "since yesterday and took 2 cold & flu tablets this morning...")
        if st.button("✨ Pre-fill the form for me", disabled=not (key and txt.strip()), key=k("ai_go")):
            try:
                with st.spinner("Reading your description..."):
                    ss.prefill = llm_intake.extract_fields(txt, key, KB)
                ss.run_id += 1
                st.rerun()
            except llm_intake.IntakeError as exc:
                st.warning(f"{exc} You can continue with the form below.")
        if not key:
            st.caption("AI pre-fill is off (no API key). The guided form below works fully without it.")


def triage_step():
    pre, p = ss.prefill, ss.patient
    if not p and not pre:
        ai_prefill_box()
    if pre:
        st.info("Some answers were pre-filled from your description. Please check every answer.")
    with st.chat_message("assistant", avatar="🩺"):
        st.markdown("**Let's start with safety.** What is the main problem, and do you have any warning signs?")
        keys = list(KB.complaints)
        default = p.get("complaint") or pre.get("complaint")
        complaint = st.selectbox("Main problem", keys, format_func=lambda c: KB.complaints[c]["label"],
                                 index=keys.index(default) if default in keys else None,
                                 placeholder="Choose the main problem", key=k("complaint"))
        days = st.number_input("How many days have you had it? (0 = started today)", min_value=0,
                               max_value=365, step=1, key=k("days"),
                               value=_int(p.get("symptom_days", pre.get("symptom_days"))))
        if complaint is None:
            return
        if ss.lang == "ur":
            st.caption(T("clinical_en_only"))
        st.markdown("**Do you have any of the following?** Please answer every question.")
        answers = {}
        prev = p.get("red_flags") or {}
        for f in KB.red_flags_for(complaint):
            v = prev.get(f["id"])
            if v is None and f["id"] in pre.get("red_flags_positive", []):
                v = True
            ans = st.radio(f["text"], YES_NO, index=yn_index(v), horizontal=True,
                           key=k(f"rf_{complaint}_{f['id']}"))
            answers[f["id"]] = None if ans is None else ans == "Yes"
        if any(answers.values()):
            st.error("⚠️ A 'Yes' answer means you need medical help rather than self-treatment.")
        if st.button("Continue →", type="primary", key=k("triage_go")):
            if days is None or None in answers.values():
                st.error("Please answer every question before continuing.")
                return
            p.update(complaint=complaint, symptom_days=int(days), red_flags=answers)
            st.rerun()


def profile_step():
    pre, p = ss.prefill, ss.patient
    with st.chat_message("assistant", avatar="🩺"):
        st.markdown("**A few details about you.** These decide whether self-care is safe and which dose fits.")
        c1, c2 = st.columns(2)
        age = c1.number_input("Age (years)", min_value=0, max_value=120, step=1, key=k("age"),
                              value=_int(p.get("age", pre.get("age"))))
        w = p.get("weight_kg", pre.get("weight_kg"))
        weight = c2.number_input("Weight (kg)", min_value=1.0, max_value=300.0, step=0.5, format="%.1f",
                                 key=k("weight"), value=float(w) if w is not None else None)
        sexes = ["female", "male", "other"]
        sdef = p.get("sex", pre.get("sex"))
        sex = st.radio("Sex", sexes, format_func=str.capitalize, horizontal=True, key=k("sex"),
                       index=sexes.index(sdef) if sdef in sexes else None)
        preg = bf = None
        if sex in ("female", "other"):
            preg = st.radio("Are you pregnant or could you be pregnant?", YES_NO, horizontal=True,
                            key=k("preg"), index=yn_index(p.get("pregnant", pre.get("pregnant"))))
            bf = st.radio("Are you breastfeeding?", YES_NO, horizontal=True, key=k("bf"),
                          index=yn_index(p.get("breastfeeding", pre.get("breastfeeding"))))
        if st.button("Continue →", type="primary", key=k("profile_go")):
            if age is None or weight is None or sex is None or (sex != "male" and (preg is None or bf is None)):
                st.error("Please answer every question before continuing.")
                return
            p.update(age=int(age), weight_kg=float(weight), sex=sex,
                     pregnant=(preg == "Yes") if sex != "male" else False,
                     breastfeeding=(bf == "Yes") if sex != "male" else False)
            st.rerun()


def safety_step():
    pre, p = ss.prefill, ss.patient
    complaint = p["complaint"]
    with st.chat_message("assistant", avatar="🩺"):
        st.markdown("**Safety checks.** Your health conditions, regular medicines and allergies "
                    "decide which medicines are safe for you. *Leave a list empty if none apply.*")
        conds = st.multiselect("Health conditions", list(CONDITION_OPTIONS), format_func=CONDITION_OPTIONS.get,
                               default=p.get("conditions", pre.get("conditions", [])), key=k("conds"),
                               placeholder="None")
        meds = st.multiselect("Regular medicines", list(MED_CATEGORY_OPTIONS),
                              format_func=MED_CATEGORY_OPTIONS.get, key=k("meds"), placeholder="None",
                              default=p.get("current_meds", pre.get("current_meds", [])))
        alls = st.multiselect("Medicine allergies", list(ALLERGY_OPTIONS), format_func=ALLERGY_OPTIONS.get,
                              default=p.get("allergies", pre.get("allergies", [])), key=k("alls"),
                              placeholder="None")
        c1, c2 = st.columns(2)
        alc = c1.radio("Do you drink alcohol regularly (most days)?", YES_NO, horizontal=True, key=k("alc"),
                       index=yn_index(p.get("alcohol_regular", pre.get("alcohol_regular"))))
        nmeds = c2.number_input("How many regular prescribed medicines do you take?", min_value=0,
                                max_value=50, step=1, key=k("nmeds"),
                                value=_int(p.get("num_regular_meds", pre.get("num_regular_meds"))))

        st.markdown("#### 🕒 Medicines already taken in the last 24 hours")
        st.warning("**Check every pack.** Many cold & flu, sinus and 'extra strength' products contain "
                   "**hidden paracetamol**. Add the paracetamol from *all* products to the total.")
        prev = {e["ingredient"]: e for e in (p.get("prior_intake") or pre.get("prior_intake") or [])}
        ingredients = []
        for d in KB.candidates_for(complaint):
            ing = KB.drugs[d]["ingredient"]
            if ing not in ingredients:
                ingredients.append(ing)
        intake, incomplete = [], False
        for ing in ingredients:
            name = next(v["display_name"] for v in KB.drugs.values() if v["ingredient"] == ing)
            e = prev.get(ing, {})
            c1, c2 = st.columns(2)
            mg = c1.number_input(f"Total {name} in last 24 h (mg) - 0 if none", min_value=0.0,
                                 max_value=20000.0, step=1.0, format="%g", key=k(f"mg_{ing}"),
                                 value=float(e["total_mg_24h"]) if e.get("total_mg_24h") is not None else None)
            hrs = c2.number_input("Hours since the last dose", min_value=0.0, max_value=24.0, step=0.5,
                                  format="%g", key=k(f"h_{ing}"), disabled=not mg,
                                  value=float(e["hours_since_last"]) if e.get("hours_since_last") is not None else None)
            if mg is None or (mg > 0 and hrs is None):
                incomplete = True
            intake.append({"ingredient": ing, "total_mg_24h": mg or 0,
                           "hours_since_last": hrs if mg else None})
        with st.expander("🧮 Help me add it up"):
            a, b = st.columns(2)
            n = a.number_input("Number of tablets/sachets taken", min_value=0, step=1, key=k("calc_n"))
            per = b.number_input("mg of the ingredient per tablet (see pack)", min_value=0.0, step=1.0,
                                 format="%g", key=k("calc_mg"))
            st.markdown(f"= **{n * per:g} mg** - add this to the total above.")

        if st.button("Run safety checks →", type="primary", key=k("safety_go")):
            if alc is None or nmeds is None or incomplete:
                st.error("Please answer every question (enter 0 mg if you took none).")
                return
            p.update(conditions=conds, current_meds=meds, allergies=alls,
                     alcohol_regular=alc == "Yes", num_regular_meds=int(nmeds), prior_intake=intake)
            st.rerun()




### Cell 6.6 — The result screens
| Function | Shown when | Looks like |
|---|---|---|
| `emergency_screen` | Emergency red flag | Full red box, large tap-to-call **1122**, the flags listed, **no medicine** |
| `urgent_screen` | Urgent red flag | Orange box, "See a doctor today" |
| `refer_screen` | Out of scope, cap reached, all drugs ruled out, or too many days | Amber card with reasons and which medicines were ruled out |
| `blocked_screen` | Validator failure | **Pulsing RED ALARM**, safe fallback message, technical details |
| `option_card` / `recommend_screen` | All checks passed | Green badge + one card per option: dose table, a "wait X hours" banner when needed, and the **24-h limit bar** coloured green (≤ 50%), amber (≤ 85%) or red |
| `report_box` | Every answer | "🚩 Report incorrect advice", which writes to `incidents.csv` |

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Result screens
# --------------------------------------------------------------------------
def report_box(res: dict):
    with st.expander("🚩 Report incorrect advice"):
        txt = st.text_area("What was wrong? Please do not include names or contact details.",
                           key=k("report_txt"))
        if st.button("Send report for clinician review", key=k("report_btn")):
            log_incident("user_report", ss.patient, res, comment=txt)
            st.success("Thank you - your report has been logged for clinician review.")


def emergency_screen(res: dict):
    items = "".join(f"<li>{esc(f['text'])}</li>" for f in res["triggered_flags"])
    st.markdown(f"""
    <div class="emergency" role="alert">
      <h2 class="{rtl().strip()}">🚨 {esc(T('emergency_title'))}</h2>
      <p class="{rtl().strip()}" style="font-size:1.15rem">{esc(T('emergency_body'))}</p>
      <div class="num"><a href="tel:{esc(num)}">📞 {esc(num)}</a></div>
      <p class="{rtl().strip()}">{esc(T('call_now'))}</p>
      <ul>{items}</ul>
    </div>""", unsafe_allow_html=True)


def urgent_screen(res: dict):
    items = "".join(f"<li>{esc(f['text'])}</li>" for f in res["triggered_flags"])
    st.markdown(f"""
    <div class="urgent" role="alert">
      <h2 class="{rtl().strip()}">🩺 {esc(T('urgent_title'))}</h2>
      <p class="{rtl().strip()}">{esc(T('urgent_body'))} <a href="tel:{esc(num)}">📞 {esc(num)}</a></p>
      <ul>{items}</ul>
    </div>""", unsafe_allow_html=True)


def refer_screen(res: dict):
    reasons = "".join(f"<li>{esc(r)}</li>" for r in res["reasons"])
    excl = "".join(f"<li><b>{esc(e['display_name'])}</b>: {esc('; '.join(e['reasons']))}</li>"
                   for e in res.get("excluded", []))
    excl_html = f"<p><b>Medicines ruled out for you:</b></p><ul>{excl}</ul>" if excl else ""
    stop = f"<p>{esc(res['stop_advice'])}</p>" if res.get("stop_advice") else ""
    st.markdown(f"""
    <div class="card refer"><h3 class="{rtl().strip()}">🧑‍⚕️ {esc(T('refer_title'))}</h3>
      <ul>{reasons}</ul>{excl_html}{stop}
      <p><i>{esc(res['disclaimer'])}</i></p></div>""", unsafe_allow_html=True)


def blocked_screen(res: dict):
    st.markdown(f'<div class="alarm" role="alert">🚨 {esc(T("alarm"))}</div>', unsafe_allow_html=True)
    st.error(res["reasons"][1])
    with st.expander("Technical details (logged for review)"):
        for f in res["failures"]:
            st.markdown(f"- `{f['check']}` - {esc(f['detail'])}")


def option_card(i: int, o: dict):
    cap, prior, dmax = o["max_24h_mg"], o["prior_mg_24h"], o["dose_max_mg"]
    total = (prior + dmax) / cap
    color = "#15803d" if total <= 0.5 else ("#d97706" if total <= 0.85 else "#b91c1c")
    rows = [("Medicine", o["display_name"]), ("Strength", f"{o['strength_mg']} mg {o['form']}"),
            ("Dose", o["dose_text"]), ("How often", o["interval_text"]),
            ("Maximum in 24 h", f"{cap:g} mg"), ("Already taken (last 24 h)", f"{prior:g} mg"),
            ("Still allowed in the next 24 h", f"{o['remaining_mg_24h']:g} mg"),
            ("How to take", o["administration"])]
    table = "".join(f"<tr><th>{esc(a)}</th><td>{esc(b)}</td></tr>" for a, b in rows)
    wait = (f'<div class="wait">⏳ Do not take the next dose for another {o["next_dose_in_h"]:g} hours.</div>'
            if o["next_dose_in_h"] > 0 else "")
    notes = "".join(f"<li>{esc(n)}</li>" for n in o["limit_notes"])
    warns = "".join(f"<li>{esc(w)}</li>" for w in o["warnings"] + o["interactions"])
    verify = ("" if o["verification_status"] == "VERIFIED" else
              '<div class="verify">🔶 Prototype data: this dose entry has not yet been verified by a pharmacist.</div>')
    st.markdown(f"""
    <div class="card">
      <h3>Option {i}: {esc(o['display_name'])}</h3>{wait}
      <table class="dose">{table}</table>
      <div class="capbar" role="img" aria-label="24-hour limit used">
        <div class="prior" style="width:{100 * prior / cap:.1f}%"></div>
        <div style="width:{100 * dmax / cap:.1f}%; background:{color}"></div></div>
      <div class="caplabel">24-hour limit: {prior:g} mg taken + up to {dmax:g} mg now = <b>{prior + dmax:g} of {cap:g} mg</b></div>
      {f'<ul>{notes}</ul>' if notes else ''}
      <p><b>⚠️ Important</b></p><ul>{warns}</ul>
      <p style="font-size:.88rem;opacity:.8"><b>Source:</b> {esc(o['source'])}</p>{verify}
    </div>""", unsafe_allow_html=True)


def recommend_screen(res: dict):
    st.markdown(f'<span class="badge-ok">✅ {esc(T("safety_ok"))}</span>', unsafe_allow_html=True)
    if len(res["options"]) > 1:
        st.markdown("Choose **one** of these options - do not take them together unless a pharmacist advises it.")
    for i, o in enumerate(res["options"], 1):
        option_card(i, o)
    if res.get("excluded"):
        st.markdown("**Ruled out for you:** " + "; ".join(
            f"{e['display_name']} ({'; '.join(e['reasons'])})" for e in res["excluded"]))
    st.info(f"🗓️ {res['stop_advice']}")
    st.markdown(f"**{res['disclaimer']}**")




### Cell 6.7 — Router and footer
* The **router** picks what to draw: a question step if the engine said `ASK`, otherwise the matching result screen.
* Under each result: **Start over**, **"I answered a question by mistake"** (on emergency screens, clears the red-flag answers),
  and the report box.
* The **footer** shows the privacy note, app version and formulary verification date. It says *"NOT YET VERIFIED"* until a pharmacist signs the formulary off.

In [ ]:
%%writefile -a app.py
# --------------------------------------------------------------------------
# Router
# --------------------------------------------------------------------------
action = result["action"]
if action == "ASK":
    {"triage": triage_step, "profile": profile_step, "safety": safety_step}[cur]()
else:
    {"EMERGENCY": emergency_screen, "URGENT_DOCTOR": urgent_screen, "REFER": refer_screen,
     "BLOCKED": blocked_screen, "RECOMMEND": recommend_screen}[action](result)
    c1, c2 = st.columns(2)
    if action in ("EMERGENCY", "URGENT_DOCTOR") and c1.button("I answered a question by mistake - review answers", key=k("review_rf")):
        ss.patient.pop("red_flags", None)
        ss.run_id += 1
        st.rerun()
    if c2.button(f"🔄 {T('start_over')}", key=k("restart_main")):
        ss.patient, ss.prefill = {}, {}
        ss.run_id += 1
        st.rerun()
    report_box(result)

vdate = KB.verification_date or "NOT YET VERIFIED"
st.markdown(f"""<div class="footer"><span class="{rtl().strip()}">🔒 {esc(T('privacy'))}</span><br>
Version {APP_VERSION} · Formulary verification date: <b>{esc(vdate)}</b> ·
Research prototype - not a medical device.</div>""", unsafe_allow_html=True)


---
# Part 7 — Unit tests (60 automatic checks)

Unit tests are small programs that check one behaviour each. If someone later edits a dose or a rule incorrectly,
a test fails and shows exactly what broke.

### Cell 7.1 — `tests/conftest.py`: shared test helpers
* `kb`: the knowledge base. `patient`: a factory that makes a complete low-risk adult, which each test then modifies.
* `intake()`: a shortcut to describe "X mg of ingredient Y taken Z hours ago".

In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from rules import default_knowledge  # noqa: E402


@pytest.fixture
def kb():
    return default_knowledge()


@pytest.fixture
def patient(kb):
    """Factory for a complete, low-risk adult patient; override fields via kwargs."""
    def make(complaint="headache", **kw):
        p = dict(complaint=complaint, symptom_days=1,
                 red_flags={f["id"]: False for f in kb.red_flags_for(complaint)},
                 age=30, sex="female", weight_kg=65, pregnant=False, breastfeeding=False,
                 conditions=[], current_meds=[], allergies=[], alcohol_regular=False,
                 num_regular_meds=0, prior_intake=[])
        p.update(kw)
        return p
    return make


def intake(ing, mg, hours):
    return [{"ingredient": ing, "total_mg_24h": mg, "hours_since_last": hours}]


### Cell 7.2 — `tests/test_rules.py`: one or more tests for every rule
Grouped by rule. Examples:
* Rule 1: a "Yes" escalates even if other questions are unanswered.
* Rule 3: a stomach ulcer excludes ibuprofen.
* Rule 4: **two products adding up to 4000 mg paracetamol** → refer.
* Rule 4: weight 45 kg → 500 mg dose and 2500 mg daily cap.
* Rule 6: no option ever contains a drug outside the formulary.

In [ ]:
%%writefile tests/test_rules.py
"""Unit tests for every decision rule in rules.py."""
from conftest import intake

from rules import DISCLAIMER, evaluate


def keys(res):
    return [o["drug_key"] for o in res["options"]]


# ---- Rule 1: red-flag triage ------------------------------------------------
def test_rule1_missing_complaint_asks(kb):
    r = evaluate({}, kb)
    assert r["action"] == "ASK" and r["missing"] == ["complaint"]


def test_rule1_emergency_flag(patient, kb):
    p = patient(); p["red_flags"]["thunderclap"] = True
    r = evaluate(p, kb)
    assert r["action"] == "EMERGENCY" and r["options"] == []


def test_rule1_urgent_flag(patient, kb):
    p = patient(); p["red_flags"]["new_over_50"] = True
    assert evaluate(p, kb)["action"] == "URGENT_DOCTOR"


def test_rule1_yes_escalates_even_if_others_unanswered(patient, kb):
    p = patient(); p["red_flags"] = {"head_injury": True}
    assert evaluate(p, kb)["action"] == "EMERGENCY"


def test_rule1_unanswered_flag_asks(patient, kb):
    p = patient(); p["red_flags"]["thunderclap"] = None
    r = evaluate(p, kb)
    assert r["action"] == "ASK" and "red_flags.thunderclap" in r["missing"]


def test_rule1_general_flag_applies_to_all_complaints(patient, kb):
    for c in kb.complaints:
        p = patient(c); p["red_flags"]["chest_pain"] = True
        assert evaluate(p, kb)["action"] == "EMERGENCY"


# ---- Rule 2: demographics and scope ----------------------------------------
def test_rule2_missing_weight_asks(patient, kb):
    r = evaluate(patient(weight_kg=None), kb)
    assert r["action"] == "ASK" and r["missing"] == ["weight_kg"] and r["rule"] == 2


def test_rule2_male_needs_no_pregnancy_answer(patient, kb):
    assert evaluate(patient(sex="male", pregnant=None, breastfeeding=None), kb)["action"] == "RECOMMEND"


def test_rule2_female_pregnancy_unknown_asks(patient, kb):
    assert evaluate(patient(pregnant=None), kb)["action"] == "ASK"


def test_rule2_child_referred(patient, kb):
    r = evaluate(patient(age=17), kb)
    assert r["action"] == "REFER" and r["scope"] == "out_of_scope"


def test_rule2_pregnant_and_breastfeeding_referred(patient, kb):
    assert evaluate(patient("fever", pregnant=True), kb)["action"] == "REFER"
    assert evaluate(patient("fever", breastfeeding=True), kb)["action"] == "REFER"


# ---- Rule 3: contraindications ---------------------------------------------
def test_rule3_missing_history_asks(patient, kb):
    r = evaluate(patient(allergies=None), kb)
    assert r["action"] == "ASK" and "allergies" in r["missing"]


def test_rule3_ulcer_excludes_ibuprofen(patient, kb):
    r = evaluate(patient(conditions=["peptic_ulcer_or_gi_bleed"]), kb)
    assert keys(r) == ["paracetamol"] and r["excluded"][0]["drug"] == "ibuprofen"


def test_rule3_interacting_medicine_excludes(patient, kb):
    assert keys(evaluate(patient(current_meds=["anticoagulant"]), kb)) == ["paracetamol"]


def test_rule3_allergy_excludes(patient, kb):
    assert keys(evaluate(patient(allergies=["paracetamol"]), kb)) == ["ibuprofen"]


def test_rule3_age_limit_excludes(patient, kb):
    assert keys(evaluate(patient(age=65), kb)) == ["paracetamol"]


def test_rule3_all_excluded_refers(patient, kb):
    r = evaluate(patient(allergies=["paracetamol"], current_meds=["anticoagulant"]), kb)
    assert r["action"] == "REFER" and r["rule"] == 3


def test_rule3_kidney_liver_out_of_scope(patient, kb):
    assert evaluate(patient(conditions=["kidney_disease"]), kb)["scope"] == "out_of_scope"
    assert evaluate(patient(conditions=["liver_disease"]), kb)["scope"] == "out_of_scope"


def test_rule3_elderly_polypharmacy_out_of_scope(patient, kb):
    assert evaluate(patient(age=75, num_regular_meds=2), kb)["action"] == "REFER"
    assert evaluate(patient(age=75, num_regular_meds=1), kb)["action"] == "RECOMMEND"


# ---- Rule 4: 24-hour cap ----------------------------------------------------
def test_rule4_prior_intake_required(patient, kb):
    assert evaluate(patient(prior_intake=None), kb)["missing"] == ["prior_intake"]


def test_rule4_hours_required_when_dose_taken(patient, kb):
    assert evaluate(patient(prior_intake=intake("paracetamol", 500, None)), kb)["action"] == "ASK"


def test_rule4_cap_reached_refers(patient, kb):
    r = evaluate(patient(prior_intake=intake("paracetamol", 4000, 3)), kb)
    assert r["action"] == "REFER" and r["scope"] == "cap_reached"


def test_rule4_remaining_allowance_limits_dose(patient, kb):
    o = evaluate(patient(prior_intake=intake("paracetamol", 3500, 5)), kb)["options"][0]
    assert o["dose_max_mg"] == 500 and o["remaining_mg_24h"] == 500


def test_rule4_hidden_paracetamol_from_two_products_is_summed(patient, kb):
    p = patient(prior_intake=intake("paracetamol", 2000, 6) + intake("paracetamol", 2000, 2))
    assert evaluate(p, kb)["action"] == "REFER"


def test_rule4_alcohol_lowers_cap(patient, kb):
    o = evaluate(patient(alcohol_regular=True), kb)["options"][0]
    assert o["max_24h_mg"] == 3000


def test_rule4_low_weight_limits_single_and_daily_dose(patient, kb):
    o = evaluate(patient(weight_kg=45), kb)["options"][0]
    assert o["dose_max_mg"] == 500 and o["max_24h_mg"] == 2500


def test_rule4_interval_wait_time(patient, kb):
    o = evaluate(patient(prior_intake=intake("paracetamol", 1000, 1.5)), kb)["options"][0]
    assert o["next_dose_in_h"] == 2.5


def test_rule4_loperamide_first_then_subsequent(patient, kb):
    assert evaluate(patient("diarrhoea"), kb)["options"][0]["dose_max_mg"] == 4
    assert evaluate(patient("diarrhoea", prior_intake=intake("loperamide", 4, 1)), kb)["options"][0]["dose_max_mg"] == 2


# ---- Rule 5: duration -------------------------------------------------------
def test_rule5_duration_exceeded_refers(patient, kb):
    r = evaluate(patient(symptom_days=4), kb)
    assert r["action"] == "REFER" and r["rule"] == 5


def test_rule5_duration_at_limit_allowed(patient, kb):
    assert evaluate(patient(symptom_days=3), kb)["action"] == "RECOMMEND"


# ---- Rule 6: output content -------------------------------------------------
def test_rule6_every_option_has_required_fields(patient, kb):
    r = evaluate(patient(), kb)
    assert r["disclaimer"] == DISCLAIMER and r["stop_advice"]
    for o in r["options"]:
        for f in ("display_name", "strength_mg", "dose_text", "interval_text", "max_24h_mg",
                  "remaining_mg_24h", "warnings", "source"):
            assert o[f] not in (None, "", [])


def test_rule6_never_outputs_drug_outside_formulary(patient, kb):
    for c in kb.complaints:
        for o in evaluate(patient(c), kb)["options"]:
            assert o["drug_key"] in kb.drugs
            assert o["dose_max_mg"] <= kb.drugs[o["drug_key"]]["single_dose_max_mg"]


### Cell 7.3 — `tests/test_validator.py`: one test per validator check
Each test takes a **correct** answer, **breaks it on purpose** (overdose, invented drug, removed disclaimer…), and confirms the
right check (V0–V13) fires. The last tests confirm that the pipeline blocks a faulty engine **and writes the incident to the CSV file**.

In [ ]:
%%writefile tests/test_validator.py
"""Unit tests for every validator check, the red-alarm pipeline and the incident log."""
import copy
import csv

import pytest
from conftest import intake

from pipeline import run_pipeline
from rules import evaluate
from validator import validate


def checks(p, r, kb):
    return {f["check"] for f in validate(p, r, kb)}


@pytest.fixture
def good(patient, kb):
    p = patient()
    return p, evaluate(p, kb)


def test_clean_recommendation_passes(good, kb):
    p, r = good
    assert validate(p, r, kb) == []


def test_V0_unknown_action(good, kb):
    p, r = good; r["action"] = "MAYBE"
    assert "V0_action" in checks(p, r, kb)


def test_V1_red_flag_not_escalated(good, kb):
    p, r = good; p["red_flags"]["thunderclap"] = True
    assert "V1_red_flag" in checks(p, r, kb)


def test_V1_triage_incomplete(good, kb):
    p, r = good; p["red_flags"]["thunderclap"] = None
    assert "V1_red_flag" in checks(p, r, kb)


def test_V2_options_on_refer(good, kb):
    p, r = good; r["action"] = "REFER"
    assert "V2_options_on_non_recommend" in checks(p, r, kb)


@pytest.mark.parametrize("change", [{"age": 16}, {"pregnant": True}, {"breastfeeding": True},
                                    {"conditions": ["kidney_disease"]}, {"age": 80, "num_regular_meds": 3},
                                    {"pregnant": None}])
def test_V3_scope(good, kb, change):
    p, r = good; p.update(change)
    assert "V3_scope" in checks(p, r, kb)


def test_V4_duration(good, kb):
    p, r = good; p["symptom_days"] = 10
    assert "V4_duration" in checks(p, r, kb)


def test_V5_missing_field(good, kb):
    p, r = good; r["options"][0]["warnings"] = []
    assert "V5_fields" in checks(p, r, kb)


def test_V6_invented_drug(good, kb):
    p, r = good
    o = copy.deepcopy(r["options"][0]); o.update(drug_key="tramadol", display_name="Tramadol")
    r["options"].append(o)
    assert "V6_formulary" in checks(p, r, kb)


def test_V6_wrong_strength(good, kb):
    p, r = good; r["options"][0]["strength_mg"] = 650
    assert "V6_formulary" in checks(p, r, kb)


def test_V7_dose_sanity(good, kb):
    p, r = good; r["options"][0]["dose_min_mg"] = 750
    assert "V7_dose_sanity" in checks(p, r, kb)


def test_V8_single_dose(good, kb):
    p, r = good; r["options"][0]["dose_max_mg"] = 1500
    assert "V8_single_dose" in checks(p, r, kb)


def test_V8_low_weight_single_dose(patient, kb):
    p = patient(); r = evaluate(p, kb)
    p["weight_kg"] = 45  # engine output computed for 65 kg -> 1000 mg now too high
    assert "V8_single_dose" in checks(p, r, kb)


def test_V9_24h_cap(patient, kb):
    p = patient(); r = evaluate(p, kb)
    p["prior_intake"] = intake("paracetamol", 3500, 6)
    assert "V9_24h_cap" in checks(p, r, kb)


def test_V10_interval(patient, kb):
    p = patient(prior_intake=intake("paracetamol", 1000, 1)); r = evaluate(p, kb)
    r["options"][0]["next_dose_in_h"] = 0
    assert "V10_interval" in checks(p, r, kb)


def test_V11_contraindication(good, kb):
    p, r = good; p["current_meds"] = ["anticoagulant"]
    assert "V11_contraindication" in checks(p, r, kb)


def test_V11_age_limit(good, kb):
    p, r = good; p["age"] = 70
    assert "V11_contraindication" in checks(p, r, kb)


def test_V12_displayed_cap(good, kb):
    p, r = good; r["options"][0]["max_24h_mg"] = 6000
    assert "V12_display" in checks(p, r, kb)


def test_V13_disclaimer(good, kb):
    p, r = good; r["disclaimer"] = "Take as needed."
    assert "V13_disclaimer" in checks(p, r, kb)


def test_pipeline_blocks_and_logs(patient, kb, tmp_path):
    def faulty_engine(p, k):
        r = evaluate(p, k); r["options"][0]["dose_max_mg"] = 3000
        return r
    log = tmp_path / "incidents.csv"
    out = run_pipeline(patient(), kb, log=True, log_file=log, engine=faulty_engine)
    assert out["action"] == "BLOCKED" and out["options"] == []
    rows = list(csv.DictReader(log.open()))
    assert rows[0]["kind"] == "validator_block" and "V8_single_dose" in rows[0]["failed_checks"]


def test_pipeline_passes_clean_output(patient, kb):
    assert run_pipeline(patient(), kb)["validator"] == "passed"


### Cell 7.4 — `tests/test_llm_intake.py`: the AI cannot inject drugs or doses
Feeds `sanitize()` a malicious AI reply containing an invented drug, a dose field, unknown conditions and a fake red flag,
and checks that all of it is dropped.

In [ ]:
%%writefile tests/test_llm_intake.py
"""The optional LLM intake can only pre-fill whitelisted fields - never drugs or doses."""
from llm_intake import sanitize


def test_sanitize_drops_unknown_keys_and_doses(kb):
    raw = {"complaint": "headache", "age": 34, "sex": "female", "recommended_drug": "tramadol",
           "dose_mg": 100, "conditions": ["kidney_disease", "made_up"],
           "prior_intake": [{"ingredient": "paracetamol", "total_mg_24h": 1000, "hours_since_last": 3},
                            {"ingredient": "codeine", "total_mg_24h": 30}],
           "red_flags_positive": ["thunderclap", "not_a_flag"]}
    out = sanitize(raw, kb)
    assert "recommended_drug" not in out and "dose_mg" not in out
    assert out["conditions"] == ["kidney_disease"]
    assert [e["ingredient"] for e in out["prior_intake"]] == ["paracetamol"]
    assert out["red_flags_positive"] == ["thunderclap"]


def test_sanitize_rejects_out_of_range_and_wrong_types(kb):
    out = sanitize({"age": 500, "weight_kg": "70", "pregnant": "no", "complaint": "cancer"}, kb)
    assert out == {}


### Cell 7.5 — Run all tests
`pytest` finds and runs every `test_*` function. **Expected: `60 passed`.** Any `F` means a failure, and pytest prints which one and why.

In [ ]:
!python -m pytest -q tests

---
# Part 8 — Evaluation with penalty scoring

### Cell 8.1 — Create `test_cases.json` (104 labelled test patients)
This cell **generates** the evaluation set instead of pasting a 100 KB file. Each case has a patient and the **expected** outcome:
`ESCALATE`, `REFER`, `ASK` or `RECOMMEND`, plus drugs that must and must not be offered.

| Category | Cases | Example |
|---|---|---|
| `red_flag` | 48 | Every red flag of every complaint, plus the general red flags |
| `normal` | 16 | A standard case for each complaint, plus boundary cases (age 18, exactly 3 days) |
| `contraindication` | 13 | Ulcer, blood thinner, asthma, allergies… |
| `cap` | 10 | Hidden paracetamol, alcohol, low weight, loperamide 8 mg… |
| `out_of_scope` | 8 | Child, pregnant, breastfeeding, kidney or liver disease, elderly polypharmacy |
| `missing_info` | 5 | Weight missing, a red flag unanswered… |
| `duration` | 4 | Headache 6 days, diarrhoea 3 days… |

> ⚠️ **Important for research use:** these expected answers were written by the developer, **not a clinician**. A clinician must
> review them, without seeing the chatbot's output, before the results can count as clinical evidence.

In [ ]:
import json

tri = json.load(open("triage.json"))

def flags(c):
    """All red-flag ids (general + complaint-specific) for a complaint."""
    return [f["id"] for f in tri["general_red_flags"] + tri["complaints"][c]["red_flags"]]

def base(c="headache", **kw):
    """A complete low-risk adult; keyword arguments override answers."""
    p = dict(complaint=c, symptom_days=1, red_flags={f: False for f in flags(c)}, age=30, sex="female",
             weight_kg=65, pregnant=False, breastfeeding=False, conditions=[], current_meds=[], allergies=[],
             alcohol_regular=False, num_regular_meds=0, prior_intake=[])
    p.update(kw)
    return p

def intake(ing, mg, h):
    return [{"ingredient": ing, "total_mg_24h": mg, "hours_since_last": h}]

cases = []
def add(cat, desc, patient, action, drugs=None, forbid=None):
    """Add one labelled case. action = expected outcome; drugs = must offer; forbid = must NOT offer."""
    n = sum(c["category"] == cat for c in cases) + 1
    cases.append({"id": f"{cat[:3].upper()}{n:02d}", "category": cat, "description": desc, "patient": patient,
                  "expected": {"action": action, "expected_drugs": drugs or [], "forbidden_drugs": forbid or []}})

# --- Red flags: every complaint-specific flag, plus the general flags on two complaints ---
for c in tri["complaints"]:
    for f in tri["complaints"][c]["red_flags"]:
        p = base(c); p["red_flags"][f["id"]] = True
        add("red_flag", f"{c}: {f['text']}", p, "ESCALATE")
for c in ("headache", "heartburn"):
    for f in tri["general_red_flags"]:
        p = base(c); p["red_flags"][f["id"]] = True
        add("red_flag", f"{c} + general: {f['text']}", p, "ESCALATE")

both = ["paracetamol", "ibuprofen"]
# --- Normal (safe) cases ---
add("normal", "Adult headache, no risks", base(), "RECOMMEND", both)
add("normal", "Adult fever", base("fever"), "RECOMMEND", both)
add("normal", "Adult back/muscle pain", base("musculoskeletal_pain"), "RECOMMEND", both)
add("normal", "Adult cold aches", base("common_cold"), "RECOMMEND", both)
add("normal", "Hay fever", base("allergic_rhinitis"), "RECOMMEND", ["cetirizine"])
add("normal", "Heartburn", base("heartburn"), "RECOMMEND", ["calcium_carbonate"])
add("normal", "Mild diarrhoea, first dose", base("diarrhoea"), "RECOMMEND", ["loperamide"])
add("normal", "Male adult headache", base(sex="male", age=40, weight_kg=78), "RECOMMEND", both)
add("normal", "Age 70 headache: paracetamol only", base(age=70), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("normal", "Weight 45 kg headache: paracetamol dose weight-limited", base(weight_kg=45), "RECOMMEND", both)
add("normal", "Age exactly 18", base(age=18), "RECOMMEND", both)
add("normal", "Paracetamol 1000 mg taken 2 h ago", base(prior_intake=intake("paracetamol", 1000, 2)), "RECOMMEND", both)
add("normal", "Duration exactly at limit (3 days)", base(symptom_days=3), "RECOMMEND", both)
add("normal", "Regular alcohol: lower paracetamol cap", base(alcohol_regular=True), "RECOMMEND", both)
add("normal", "Age 76 with 1 regular medicine", base(age=76, num_regular_meds=1), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("normal", "Ibuprofen 400 mg taken 2 h ago", base(prior_intake=intake("ibuprofen", 400, 2)), "RECOMMEND", both)
# --- Contraindications ---
add("contraindication", "Back pain + stomach ulcer history", base("musculoskeletal_pain", conditions=["peptic_ulcer_or_gi_bleed"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Headache on warfarin", base(current_meds=["anticoagulant"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Fever + NSAID-sensitive asthma", base("fever", conditions=["asthma_nsaid_sensitive"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Headache + paracetamol allergy", base(allergies=["paracetamol"]), "RECOMMEND", ["ibuprofen"], ["paracetamol"])
add("contraindication", "Paracetamol allergy + anticoagulant", base(allergies=["paracetamol"], current_meds=["anticoagulant"]), "REFER", [], both)
add("contraindication", "Muscle pain, already on diclofenac", base("musculoskeletal_pain", current_meds=["nsaid"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Hay fever + cetirizine allergy", base("allergic_rhinitis", allergies=["cetirizine_hydroxyzine"]), "REFER", [], ["cetirizine"])
add("contraindication", "Headache + heart failure", base(conditions=["heart_failure"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Diarrhoea + ulcerative colitis", base("diarrhoea", conditions=["inflammatory_bowel_disease"]), "REFER", [], ["loperamide"])
add("contraindication", "Headache on sertraline", base(current_meds=["ssri_snri"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Muscle pain + high blood pressure", base("musculoskeletal_pain", conditions=["high_blood_pressure"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
add("contraindication", "Heartburn + antacid allergy", base("heartburn", allergies=["calcium_carbonate"]), "REFER", [], ["calcium_carbonate"])
add("contraindication", "Ibuprofen + aspirin allergy, fever", base("fever", allergies=["nsaid_aspirin"]), "RECOMMEND", ["paracetamol"], ["ibuprofen"])
# --- 24-hour cap ---
add("cap", "Hidden paracetamol: 4000 mg from cold & flu products", base(prior_intake=intake("paracetamol", 4000, 3)), "REFER", [], ["paracetamol"])
add("cap", "3500 mg paracetamol taken: only 500 mg left", base(prior_intake=intake("paracetamol", 3500, 5)), "RECOMMEND", both)
add("cap", "Cold: 3000 mg from combination product", base("common_cold", prior_intake=intake("paracetamol", 3000, 4)), "RECOMMEND", both)
add("cap", "Regular alcohol + 3000 mg already", base(alcohol_regular=True, prior_intake=intake("paracetamol", 3000, 5)), "REFER", [], ["paracetamol"])
add("cap", "45 kg + 2500 mg already (weight cap reached)", base(weight_kg=45, prior_intake=intake("paracetamol", 2500, 5)), "REFER", [], ["paracetamol"])
add("cap", "Loperamide 8 mg already", base("diarrhoea", prior_intake=intake("loperamide", 8, 1)), "REFER", [], ["loperamide"])
add("cap", "Ibuprofen 1200 mg already", base("musculoskeletal_pain", prior_intake=intake("ibuprofen", 1200, 6)), "REFER", [], ["ibuprofen"])
add("cap", "Loperamide 6 mg already: 2 mg left", base("diarrhoea", prior_intake=intake("loperamide", 6, 1)), "RECOMMEND", ["loperamide"])
add("cap", "Fever, 3600 mg paracetamol (less than one tablet left)", base("fever", prior_intake=intake("paracetamol", 3600, 4)), "REFER", [], ["paracetamol"])
add("cap", "Two products adding to 4000 mg paracetamol", base(prior_intake=[{"ingredient": "paracetamol", "total_mg_24h": 2000, "hours_since_last": 6}, {"ingredient": "paracetamol", "total_mg_24h": 2000, "hours_since_last": 2}]), "REFER", [], ["paracetamol"])
# --- Out of scope ---
add("out_of_scope", "Age 16 headache", base(age=16), "REFER", [], both)
add("out_of_scope", "Age 10 fever", base("fever", age=10, weight_kg=30), "REFER", [], both)
add("out_of_scope", "Pregnant, back pain", base("musculoskeletal_pain", pregnant=True), "REFER", [], both)
add("out_of_scope", "Breastfeeding, cold", base("common_cold", breastfeeding=True), "REFER", [], both)
add("out_of_scope", "Chronic kidney disease, headache", base(conditions=["kidney_disease"]), "REFER", [], both)
add("out_of_scope", "Liver disease, fever", base("fever", conditions=["liver_disease"]), "REFER", [], both)
add("out_of_scope", "Age 80 with 3 regular medicines", base("musculoskeletal_pain", age=80, num_regular_meds=3), "REFER", [], both)
add("out_of_scope", "Pregnant, heartburn", base("heartburn", pregnant=True), "REFER", [], ["calcium_carbonate"])
# --- Missing information ---
p = base(); p["weight_kg"] = None
add("missing_info", "Weight not given", p, "ASK", [], both)
p = base(); p["red_flags"]["thunderclap"] = None
add("missing_info", "One red-flag question unanswered", p, "ASK", [], both)
p = base(); p["prior_intake"] = None
add("missing_info", "Prior intake not asked", p, "ASK", [], both)
add("missing_info", "Paracetamol taken but time unknown", base(prior_intake=intake("paracetamol", 1000, None)), "ASK", [], both)
p = base(); p["pregnant"] = None
add("missing_info", "Pregnancy status unknown (female)", p, "ASK", [], both)
# --- Duration ---
add("duration", "Headache for 6 days", base(symptom_days=6), "REFER", [], both)
add("duration", "Diarrhoea for 3 days", base("diarrhoea", symptom_days=3), "REFER", [], ["loperamide"])
add("duration", "Heartburn for 20 days", base("heartburn", symptom_days=20), "REFER", [], ["calcium_carbonate"])
add("duration", "Cold for 10 days", base("common_cold", symptom_days=10), "REFER", [], both)

doc = {"_meta": {"version": "0.1.0",
                 "gold_status": "DRAFT - expected outcomes written by the developer; must be reviewed and signed off by a clinician before being used as gold labels.",
                 "actions": "ESCALATE = EMERGENCY or URGENT_DOCTOR; REFER includes validator BLOCKED"},
       "cases": cases}
json.dump(doc, open("test_cases.json", "w"), indent=1)

from collections import Counter
print(f"✅ Wrote {len(cases)} test cases:", dict(Counter(c["category"] for c in cases)))

### Cell 8.2 — `evaluate.py`: the scoring program
Runs every test case **twice**, first through the engine alone and then through engine + validator (what users actually see), and scores mistakes:

| Mistake | Penalty | Critical? |
|---|---|---|
| Medicine suggested although a red flag was present | **−100** | ✅ |
| Dose above the single or 24-hour maximum | **−100** | ✅ |
| Contraindicated drug suggested | −80 | |
| Out-of-scope, cap-reached or too-long case given a dose | −60 | |
| Dose given although information was missing | −60 | |
| Red flag referred instead of escalated | −20 | |
| Safe case referred unnecessarily (over-caution) | −5 | |
| A safe first-line option not offered | −5 | |
| Missing warning or display field | −2 | |

It reports total penalty, **critical-error count (target 0)**, **red-flag sensitivity** (share of red-flag cases correctly escalated),
**over-referral rate** (share of safe cases needlessly referred), action accuracy, and a **confusion table**.

**Fault injection** (`--inject-faults`) feeds the validator deliberately broken outputs: tripled doses, an invented drug ("codeine"),
a missed blood-thinner interaction, removed warnings, a removed disclaimer, ignored earlier doses, and medicines suggested for red-flag
patients. It then measures what share the validator catches.

The program **exits with code 1 if the deployed pipeline makes any critical error**. That makes it usable as an automatic safety gate before every release.

In [ ]:
%%writefile evaluate.py
"""
evaluate.py - Offline safety evaluation with weighted penalty scoring.

Usage:
    python evaluate.py                       # score the deployed pipeline
    python evaluate.py --inject-faults       # also stress-test the validator
    python evaluate.py --json-out report.json

Each case is scored twice:
  * "engine"   - raw rules-engine output (no validator)
  * "pipeline" - engine + safety validator, i.e. what a user would see
The build FAILS (exit code 1) if the pipeline makes any critical error.

Penalties (per event)
  missed_red_flag            -100  critical  dosed a red-flag case
  dose_above_max             -100  critical  single or 24-h maximum exceeded
  contraindicated_drug        -80            forbidden drug offered
  dosed_should_refer          -60            out-of-scope / cap / duration case dosed
  dosed_missing_info          -60            dosed although required info missing
  under_escalation            -20            red flag referred/asked instead of escalated (*)
  unnecessary_referral         -5            safe case not dosed
  missing_expected_option      -5            a safe first-line option was not offered (*)
  missing_field                -2            option lacks a required display field
  wrong_non_dosing_action      -2            e.g. ASK instead of REFER (*)
(*) additions to the original specification, documented in README.
"""
from __future__ import annotations

import argparse
import copy
import json
import sys
from collections import Counter, defaultdict
from pathlib import Path

from pipeline import run_pipeline
from rules import Knowledge, evaluate as engine_evaluate
from validator import REQUIRED_OPTION_FIELDS, patient_limits

PENALTY = {
    "missed_red_flag": (-100, True), "dose_above_max": (-100, True),
    "contraindicated_drug": (-80, False), "dosed_should_refer": (-60, False),
    "dosed_missing_info": (-60, False), "under_escalation": (-20, False),
    "unnecessary_referral": (-5, False), "missing_expected_option": (-5, False),
    "missing_field": (-2, False), "wrong_non_dosing_action": (-2, False),
}
CLASSES = ["ESCALATE", "REFER", "ASK", "RECOMMEND"]


def classify(action: str) -> str:
    return {"EMERGENCY": "ESCALATE", "URGENT_DOCTOR": "ESCALATE",
            "BLOCKED": "REFER"}.get(action, action)


def score_case(case: dict, out: dict, kb: Knowledge) -> list[tuple[str, int, bool, str]]:
    exp = case["expected"]
    want, got = exp["action"], classify(out["action"])
    p = case["patient"]
    events = []

    def ev(name, detail=""):
        pts, crit = PENALTY[name]
        events.append((name, pts, crit, detail))

    if got == "RECOMMEND":
        offered = [o.get("drug_key") for o in out.get("options", [])]
        for o in out.get("options", []):
            drug = kb.drugs.get(o.get("drug_key"))
            for f in REQUIRED_OPTION_FIELDS:
                if o.get(f) in (None, "", []):
                    ev("missing_field", f"{o.get('drug_key')}.{f}")
            if drug is None:
                ev("contraindicated_drug", f"unknown drug {o.get('drug_key')}")
                continue
            single_max, max24 = patient_limits(drug, p)
            prior = sum(e.get("total_mg_24h") or 0 for e in p.get("prior_intake") or []
                        if e.get("ingredient") == drug["ingredient"])
            dmax = o.get("dose_max_mg") or 0
            if dmax > single_max or prior + dmax > max24:
                ev("dose_above_max", f"{o['drug_key']}: {prior:g}+{dmax} mg (single {single_max:g}, 24h {max24:g})")
        for d in offered:
            if d in exp.get("forbidden_drugs", []):
                ev("contraindicated_drug", d)
        if want == "ESCALATE":
            ev("missed_red_flag")
        elif want == "REFER":
            ev("dosed_should_refer")
        elif want == "ASK":
            ev("dosed_missing_info")
        else:
            for d in exp.get("expected_drugs", []):
                if d not in offered:
                    ev("missing_expected_option", d)
    else:
        if want == "RECOMMEND":
            ev("unnecessary_referral", got)
        elif want == "ESCALATE" and got != "ESCALATE":
            ev("under_escalation", got)
        elif want != got:
            ev("wrong_non_dosing_action", f"expected {want}, got {got}")
    return events


def run(cases: list[dict], kb: Knowledge, use_validator: bool) -> dict:
    total, critical, rows = 0, 0, []
    confusion = defaultdict(Counter)
    blocked = 0
    for c in cases:
        out = run_pipeline(c["patient"], kb) if use_validator else engine_evaluate(c["patient"], kb)
        blocked += out["action"] == "BLOCKED"
        events = score_case(c, out, kb)
        pen = sum(e[1] for e in events)
        crit = any(e[2] for e in events)
        total += pen
        critical += sum(1 for e in events if e[2])
        confusion[c["expected"]["action"]][classify(out["action"])] += 1
        rows.append({"id": c["id"], "category": c["category"], "expected": c["expected"]["action"],
                     "got": out["action"], "penalty": pen, "critical": crit,
                     "events": [f"{e[0]}({e[1]}) {e[3]}".strip() for e in events]})
    n_rf = sum(confusion["ESCALATE"].values())
    n_safe = sum(confusion["RECOMMEND"].values())
    correct = sum(confusion[a][a] for a in CLASSES)
    return {
        "total_penalty": total, "critical_errors": critical, "validator_blocks": blocked,
        "action_accuracy": correct / len(cases),
        "red_flag_sensitivity": confusion["ESCALATE"]["ESCALATE"] / n_rf if n_rf else None,
        "over_referral_rate": (confusion["RECOMMEND"]["REFER"] + confusion["RECOMMEND"]["ESCALATE"]) / n_safe if n_safe else None,
        "confusion": {a: dict(confusion[a]) for a in CLASSES},
        "cases": rows,
    }


# --------------------------------------------------------------------------
# Fault injection: does the validator catch a faulty engine?
# --------------------------------------------------------------------------
def _mutations(kb: Knowledge):
    def overdose(r, p):
        for o in r["options"]:
            o["dose_max_mg"] = o["dose_max_mg"] * 3
        return r

    def invented_drug(r, p):
        o = copy.deepcopy(r["options"][0])
        o.update(drug_key="codeine", display_name="Codeine")
        r["options"].append(o)
        return r

    def ignore_contraindication(r, p):
        p.setdefault("current_meds", []).append("anticoagulant")
        return r  # patient changed after engine ran -> ibuprofen now unsafe if offered

    def drop_warnings(r, p):
        for o in r["options"]:
            o["warnings"] = []
        return r

    def drop_disclaimer(r, p):
        r["disclaimer"] = ""
        return r

    def ignore_prior_intake(r, p):
        for o in r["options"]:
            p.setdefault("prior_intake", []).append(
                {"ingredient": kb.drugs[o["drug_key"]]["ingredient"],
                 "total_mg_24h": kb.drugs[o["drug_key"]]["max_24h_mg"], "hours_since_last": 1})
        return r

    return {"overdose_x3": overdose, "invented_drug": invented_drug,
            "missed_interaction": ignore_contraindication, "drop_warnings": drop_warnings,
            "drop_disclaimer": drop_disclaimer, "ignored_prior_intake": ignore_prior_intake}


def fault_injection(cases: list[dict], kb: Knowledge) -> dict:
    report = {}
    base_cases = [c for c in cases if engine_evaluate(c["patient"], kb)["action"] == "RECOMMEND"]
    for name, mutate in _mutations(kb).items():
        caught = applicable = 0
        for c in base_cases:
            p = copy.deepcopy(c["patient"])
            r = mutate(copy.deepcopy(engine_evaluate(p, kb)), p)
            if name == "missed_interaction" and not any(o["drug_key"] == "ibuprofen" for o in r["options"]):
                continue
            applicable += 1
            out = run_pipeline(p, kb, engine=lambda _p, _kb, r=r: r)
            caught += out["action"] == "BLOCKED"
        report[name] = {"applicable": applicable, "caught": caught,
                        "catch_rate": caught / applicable if applicable else None}
    # red flag escalation bypass: engine wrongly recommends for red-flag cases
    rf = [c for c in cases if c["expected"]["action"] == "ESCALATE"]
    safe = engine_evaluate(next(c for c in cases if c["expected"]["action"] == "RECOMMEND"
                                and c["patient"]["complaint"] == "headache")["patient"], kb)
    caught = 0
    for c in rf:
        fake = copy.deepcopy(safe)
        out = run_pipeline(c["patient"], kb, engine=lambda _p, _kb, r=fake: r)
        caught += out["action"] == "BLOCKED"
    report["red_flag_bypass"] = {"applicable": len(rf), "caught": caught, "catch_rate": caught / len(rf)}
    return report


def _pct(x):
    return "n/a" if x is None else f"{100 * x:.1f}%"


def print_report(name: str, r: dict):
    print(f"\n=== {name} ===")
    print(f"Total penalty        : {r['total_penalty']}")
    print(f"Critical errors      : {r['critical_errors']}  (target = 0)")
    print(f"Red-flag sensitivity : {_pct(r['red_flag_sensitivity'])}")
    print(f"Over-referral rate   : {_pct(r['over_referral_rate'])}")
    print(f"Action accuracy      : {_pct(r['action_accuracy'])}")
    print(f"Validator blocks     : {r['validator_blocks']}")
    print("Confusion (rows = expected, cols = predicted):")
    print("            " + "".join(f"{c:>11}" for c in CLASSES))
    for a in CLASSES:
        print(f"{a:>11} " + "".join(f"{r['confusion'][a].get(b, 0):>11}" for b in CLASSES))
    bad = [c for c in r["cases"] if c["penalty"]]
    if bad:
        print("Penalised cases:")
        for c in bad:
            print(f"  {c['id']:<6} {c['expected']:>9} -> {c['got']:<13} {c['penalty']:>5}  {'; '.join(c['events'])}")


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[1])
    ap.add_argument("--cases", default=str(Path(__file__).with_name("test_cases.json")))
    ap.add_argument("--inject-faults", action="store_true")
    ap.add_argument("--json-out")
    a = ap.parse_args(argv)
    cases = json.loads(Path(a.cases).read_text(encoding="utf-8"))["cases"]
    kb = Knowledge()
    print(f"Loaded {len(cases)} cases: {dict(Counter(c['category'] for c in cases))}")
    report = {"engine": run(cases, kb, use_validator=False),
              "pipeline": run(cases, kb, use_validator=True)}
    print_report("Engine only (no validator)", report["engine"])
    print_report("Deployed pipeline (engine + validator)", report["pipeline"])
    if a.inject_faults:
        report["fault_injection"] = fault_injection(cases, kb)
        print("\n=== Fault injection: validator catch rate ===")
        for k, v in report["fault_injection"].items():
            print(f"  {k:<22} caught {v['caught']}/{v['applicable']}  ({_pct(v['catch_rate'])})")
    if a.json_out:
        Path(a.json_out).write_text(json.dumps(report, indent=2), encoding="utf-8")
    failed = report["pipeline"]["critical_errors"] > 0
    print("\nBUILD", "FAILED - critical safety errors" if failed else "PASSED - no critical errors")
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())


### Cell 8.3 — Run the evaluation
**Expected:** total penalty 0, critical errors 0, red-flag sensitivity 100%, over-referral 0%, every fault caught (100%),
and `BUILD PASSED`.

**How to interpret this honestly:** a perfect score here shows the **code matches its specification**. It does **not** prove clinical
safety, because the same person wrote the rules and the expected answers. For a publishable result, use clinician-written labels and real pharmacy queries.

In [ ]:
!python evaluate.py --inject-faults

---
# Part 9 — Launch the chatbot 🚀

### Cell 9.1 — Optional: enable AI pre-fill
**Skip this cell if you have no API key**, because the chatbot works fully without one. To enable it:
1. Click the 🔑 **Secrets** icon in the left sidebar.
2. Add a secret named `ANTHROPIC_API_KEY`.
3. Switch **Notebook access** on.

The `try/except` means this cell never stops the notebook with an error.

In [ ]:
import os
try:
    from google.colab import userdata                        # Colab's secure key store
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("✅ AI pre-fill enabled")
except Exception:
    print("ℹ️ No API key found — the app will use the guided form only (this is fine).")

### Cell 9.2 — Start the app and create a public link
Colab cannot show web apps directly, so this cell does three things:

1. **Starts Streamlit in the background** on port 8501. The output goes to `streamlit.log`.
   `app_ok()` asks the app's health endpoint (`/_stcore/health`) whether it is running.
2. **Starts a free Cloudflare "quick tunnel"** (`cloudflared`, no account needed). It creates a public `https://….trycloudflare.com`
   address that forwards to the app. `--protocol http2` is used because the default (QUIC) can stall in Colab.
3. **Reads `tunnel.log` until the link appears** (up to 2 minutes), then shows it as a big green clickable link.

`pkill` first stops any old copies, so re-running this cell is always safe.

> 🔗 Anyone with the link can use the app while this notebook is running. Share it only with people you choose.
> The link dies when Colab disconnects. Re-run this cell for a new one.

In [ ]:
import os, re, time, subprocess, urllib.request
from IPython.display import HTML, display
os.chdir(PROJECT_DIR)

def app_ok():
    """True if the Streamlit app answers on its health endpoint."""
    try:
        return urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=3).read() == b"ok"
    except Exception:
        return False

# 1) (Re)start the chatbot app in the background
subprocess.run(["pkill", "-f", "streamlit run"], capture_output=True)
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                 stdout=open("streamlit.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):                        # wait up to 30 s for the app to start
    time.sleep(1)
    if app_ok():
        break
print("Chatbot app running:", app_ok())

# 2) Download cloudflared once, then start a free tunnel to the app
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)
if not os.path.exists("cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared
subprocess.Popen(["./cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2",
                  "--url", "http://localhost:8501"],
                 stdout=open("tunnel.log", "w"), stderr=subprocess.STDOUT)

# 3) Wait for the public link to appear in the tunnel log
url = None
for _ in range(60):                        # wait up to 2 minutes
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        url = m.group(0)
        break

if url:
    time.sleep(5)                          # give the tunnel a moment to become reachable
    display(HTML(f'<h2>✅ Your chatbot is ready: <a href="{url}" target="_blank">{url}</a></h2>'))
else:
    print("❌ No link yet. Run Cell 9.3 (backup). Last lines of the tunnel log:")
    print("".join(open("tunnel.log").readlines()[-30:]))

### Cell 9.3 — Backup link (only if Cell 9.2 showed ❌)
Uses a different free tunnel service, **localtunnel**.
1. The cell prints a **password**, which is your Colab machine's public IP address, and a `https://….loca.lt` link.
2. Open the link, paste the password, and click **Click to Submit**.

In [ ]:
import subprocess, time, urllib.request
!npm install -g localtunnel > /dev/null 2>&1
print("PASSWORD for the page:", urllib.request.urlopen("https://loca.lt/mytunnelpassword").read().decode())
subprocess.Popen("lt --port 8501 > lt.log 2>&1", shell=True)
time.sleep(10)
print(open("lt.log").read())

---
# Part 10 — Troubleshooting, downloads and limitations

### Cell 10.1 — Show the logs
Use this when the page does not load. The first block is the app (look for Python errors), and the second is the tunnel.
`Uvicorn server started` means the app itself is fine.

In [ ]:
!echo "===== streamlit.log ====="; tail -20 streamlit.log
!echo "===== tunnel.log =====";    tail -20 tunnel.log

### Cell 10.2 — Download the incident log
Downloads `incidents.csv`: validator blocks and "Report incorrect advice" submissions. **Colab deletes files when the runtime ends**, so download it before closing.
*"No incidents logged yet."* is normal and simply means nothing has gone wrong.

In [ ]:
import os
from google.colab import files
p = os.path.join(PROJECT_DIR, "incidents.csv")
files.download(p) if os.path.exists(p) else print("No incidents logged yet.")

### Cell 10.3 — `README.md`: project documentation
The README goes inside the downloaded project, so the project explains itself on GitHub: setup, testing, deployment, how to update the formulary, and limitations.

In [ ]:
%%writefile README.md
# OTC Safe-Dose Assistant

A safety-first, rule-based chatbot that gives adults general guidance on
over-the-counter (OTC) medicines for common, self-limiting complaints:
headache, fever, mild muscle/joint pain, common cold, hay fever, heartburn
and mild diarrhoea.

> **Research prototype. Not a medical device. Not clinically validated.**
> Every dose in `formulary.json` is a placeholder marked `VERIFY` until a
> pharmacist checks it against the official product label.

## Design in one picture

```
patient answers ─► rules.py (Rules 1-6) ─► validator.py (V0-V13) ─► GUI
   (form, or LLM         deterministic,         independent re-check;     shows answer, or
    pre-fill that the    reads formulary.json   any failure = BLOCK       RED ALARM + incident
    user confirms)       and triage.json                                  logged to incidents.csv
```

* **The LLM never chooses a drug or a dose.** It is optional and only
  pre-fills the form from free text; the user confirms every answer.
  It can pre-mark a red flag as "Yes" but never as "No".
* **The rules engine** (`rules.py`) runs strictly in order and stops at the first exit:
  1. Red-flag triage → emergency / see a doctor today
  2. Demographics and scope (adults only; pregnancy/breastfeeding referred)
  3. Contraindications (conditions, interacting medicines, allergies, age)
  4. 24-hour cap, including hidden paracetamol in combination products, and the minimum interval
  5. Self-care duration limit
  6. Recommendation card with dose, interval, cap, remaining allowance, warnings, source and disclaimer
* **The safety validator** (`validator.py`) re-implements the safety
  arithmetic independently and blocks any output that breaks a hard
  constraint. Blocked outputs show a red alarm and are logged.

## Project structure

```
otc-safe-dose/
├── app.py              Streamlit GUI (stepper, cards, cap bar, emergency/alarm screens, EN/UR)
├── rules.py            Deterministic decision engine (Rules 1-6)
├── validator.py        Independent safety validator (V0-V13)
├── pipeline.py         engine -> validator -> result or BLOCKED
├── incident_log.py     CSV log for validator blocks and user reports
├── llm_intake.py       Optional free-text pre-fill (whitelisted fields only)
├── i18n.py             English / Urdu interface strings
├── formulary.json      Drugs, doses, limits, contraindications (VERIFY placeholders)
├── triage.json         Red flags per complaint, self-care limits, emergency number
├── test_cases.json     104 labelled evaluation cases (draft gold labels)
├── evaluate.py         Penalty-weighted evaluation and fault injection
├── tests/              pytest unit tests (rules, validator, LLM sanitiser)
├── requirements.txt
└── .streamlit/config.toml
```

## Setup and running

```bash
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install -r requirements.txt
streamlit run app.py
```

Optional settings (environment variables):

| Variable | Purpose | Default |
|---|---|---|
| `ANTHROPIC_API_KEY` | Enables the optional free-text pre-fill | not set (form only) |
| `MEDBOT_LLM_MODEL` | Model used for pre-fill | `claude-haiku-5-5` |
| `MEDBOT_EMERGENCY_NUMBER` | Number shown on emergency screens | `1122` |
| `MEDBOT_INCIDENT_LOG` | Path of the incident CSV | `./incidents.csv` |

## Testing and evaluation

```bash
python -m pytest -q tests               # 60 unit tests: every rule and every validator check
python evaluate.py --inject-faults      # penalty scoring + validator stress test
```

`evaluate.py` scores each case twice: the raw engine and the deployed
pipeline (engine plus validator). It reports total penalty, critical-error
count, red-flag sensitivity, over-referral rate, action accuracy and a
confusion table. It exits with code 1 if the pipeline makes any critical
error, so you can use it as a CI gate.

| Event | Penalty | Critical |
|---|---|---|
| Missed red flag (dosed instead of escalating) | -100 | yes |
| Dose above single or 24-h maximum | -100 | yes |
| Contraindicated drug recommended | -80 | |
| Out-of-scope / cap-reached / over-duration case dosed | -60 | |
| Dosed with required information missing* | -60 | |
| Red flag referred instead of escalated* | -20 | |
| Unnecessary referral of a safe case | -5 | |
| Safe first-line option not offered* | -5 | |
| Missing warning or field | -2 | |
| Wrong non-dosing action (e.g., ASK instead of REFER)* | -2 | |

\* Additions to the original specification.

**Fault injection** (`--inject-faults`) feeds deliberately faulty engine
outputs to the validator: tripled doses, an invented drug, a missed
interaction, dropped warnings or disclaimer, ignored prior intake, and a
recommendation issued for red-flag cases. It reports the catch rate for each.

### How to read the current results

The current scores (0 penalty, 100% red-flag sensitivity, 100% catch rate)
show that the code does what the labels say. They do **not** show that the
system is clinically safe. The expected outcomes in `test_cases.json` were
written by the developer from the same rules, so agreement is partly
circular. For a meaningful result:

1. Have one or more clinicians write or review the gold labels without seeing the engine's output.
2. Add cases drawn from real pharmacy queries, including ambiguous ones.
3. Report inter-rater agreement between clinicians (for example, Cohen's kappa).
4. Consider a user study of whether people answer the triage questions correctly. Incorrect self-report is a major real-world failure mode that this evaluation cannot capture.

## Deployment (free)

**Streamlit Community Cloud:** push this folder to a public GitHub repository,
sign in at share.streamlit.io, choose the repository and `app.py`, and
optionally add `ANTHROPIC_API_KEY` under *Secrets*.

**Hugging Face Spaces:** create a new Space, choose the Streamlit or Docker
option offered there, and upload these files. Follow the current Spaces
documentation, because the available templates change over time.

Both platforms use temporary file systems. `incidents.csv` can be lost when
the app restarts. For real incident tracking, write to a persistent store,
such as a database or a private dataset repository.

## Updating the formulary

1. Edit the entry in `formulary.json` using the exact product you dispense (strength, form, label dose, interval, 24-h maximum, contraindications, interactions, warnings).
2. Fill in `source`, set `verified_by` and `verified_date`, and change `status` to `VERIFIED`. Set `_meta.formulary_verification_date`.
3. To add a drug, also list it under the complaint's `candidates` in `triage.json`. Any new contraindication keys must exist in `CONDITION_OPTIONS`, `MED_CATEGORY_OPTIONS` or `ALLERGY_OPTIONS` in `rules.py`.
4. Add test cases for the new entry, then run `pytest` and `python evaluate.py`. Deploy only if both pass.

## Limitations

* **Not a medical device** and not clinically validated. It does not diagnose.
* **Formulary and red flags are unverified placeholders.** A pharmacist and a clinician must review them before any public use.
* **Regulatory approval is required before real patient use.** Software that recommends doses to individuals may be regulated as Software as a Medical Device (SaMD). Check the requirements of the Drug Regulatory Authority of Pakistan (DRAP), or the authority where you deploy.
* Adults only, OTC only, seven complaints only. Children, pregnancy, breastfeeding, chronic kidney/liver disease and older adults on several medicines are referred, not dosed.
* Safety depends on honest, accurate self-report. The app cannot examine the patient.
* Medicine categories are coarse. Uncommon interactions may be missed, so "Other regular medicine" is a prompt to ask a pharmacist, not a check.
* When one ingredient's 24-hour cap is reached, the app refers the user instead of suggesting a different medicine. This is a deliberately conservative choice.
* Clinical questions are English-only. The Urdu toggle translates the interface but not the medical questions, pending professional clinical translation.


### Cell 10.4 — Download the whole project as a zip
Packs every file this notebook created, so you can put it on GitHub or deploy it to Streamlit Community Cloud.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive("/content/otc-safe-dose", "zip", PROJECT_DIR)
files.download(zip_path)

---
## ⚠️ Limitations (read before showing or publishing)

* **Not a medical device** and not clinically validated. It does not diagnose.
* **Doses and red flags are unverified placeholders.** A pharmacist and a clinician must review `formulary.json` and `triage.json`.
* **Regulatory approval** (for example, from the Drug Regulatory Authority of Pakistan, DRAP) would be needed before real patients use it.
  Software that recommends doses to individuals may count as Software as a Medical Device (SaMD).
* **Evaluation labels are developer-written.** The 100% scores show the code matches its specification, not that it is clinically safe.
* **Self-report risk:** the app trusts the user's answers. One known weakness: amounts taken are entered in **mg**, so a user
  who types "2" meaning *2 tablets* is read as 2 mg. Asking for *number of tablets × mg per tablet* is a recommended next improvement.
* **"Other regular medicine"** cannot be checked for interactions. Such users should be advised to ask a pharmacist.
* Adults only, 5 OTC medicines, 7 complaints. Medical questions are in English only.

## 🔭 Suggested next steps
1. Ask for the amount taken as tablets × strength, and reject implausible amounts.
2. Show a "pharmacist check needed" note whenever "Other regular medicine" is selected.
3. Have the formulary verified and the 104 expected answers reviewed by clinicians, and report inter-rater agreement.
4. Run a small usability study to measure how accurately people answer the triage questions themselves.